# 09. Portfolio Performance & Sensitivity Analysis

Una vez calculadas las rentabilidades netas tras descontar el impacto de la fricción operativa y el turnover en el notebook anterior, se procede a la evaluación económica final de las carteras. Este bloque tiene como objetivo analizar en profundidad el rendimiento ajustado al riesgo de las distintas estrategias, evaluar la estabilidad temporal de las métricas en diferentes subperiodos OOS, estudiar la sensibilidad de los resultados ante variaciones en los parámetros clave de ejecución y establecer la comparativa definitiva entre los modelos de Machine Learning y los benchmarks de referencia.

## 1. Imports & Configuration

### 1.1 Librerías

In [1]:
import sys
import pandas as pd 
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path
from scipy.stats import spearmanr

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

### 1.2 Configuración del notebook

In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

PREDICTION_HORIZON = 21

TRADING_DAYS_PER_YEAR = 252

RISK_FREE_RATE = 0.0

REBALANCING_FREQUENCY = 21

BASE_TRANSACTION_COST = 0.0015 


### 1.3 Rutas y parámetros globales

In [3]:
# =============================================================================
# Paths
# =============================================================================

PRICES_PATH = "../data/raw/sp500_prices_extended.parquet"

RAW_WEIGHTS_PATH = "../data/portfolio_results/final_portfolio_weights.parquet"

NET_RETURNS = "../data/portfolio_results/net_portfolio_returns.parquet"

## 2. Data Loading

In [4]:
# =============================================================================
# Load Extended Price Data
# =============================================================================

prices = pd.read_parquet("../data/raw/sp500_prices_extended.parquet")

# =============================================================================
# Compute Daily Asset Returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

# =============================================================================
# Load Net Portfolio Returns
# =============================================================================

net_returns = pd.read_parquet(
    "../data/portfolio_results/net_portfolio_returns.parquet"
)


# =============================================================================
# Load Final Portfolio Weights
# =============================================================================

weights_raw = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

## 3. Portfolio Performance & Sensitivity Analysis

Una vez obtenidas las rentabilidades netas de las distintas estrategias, se realiza la evaluación económica final de las carteras. Este bloque tiene tres objetivos: comparar el rendimiento ajustado al riesgo de las estrategias, estudiar la sensibilidad de los resultados ante cambios en la frecuencia de rebalanceo y establecer una comparación final entre los modelos de Machine Learning y las estrategias de referencia.


## 3. Global Net Performance Evaluation

Tras someter las estrategias a las fricciones de ejecución en el último capítulo del notebook 08, la evaluación final de rendimiento debe realizarse de forma estricta sobre las **rentabilidades netas** de comisiones. Medir el valor generado únicamente mediante la rentabilidad acumulada resulta insuficiente en la gestión cuantitativa moderna: es imprescindible contrastar el retorno obtenido contra la volatilidad asumida, la severidad de las caídas patrimoniales y la velocidad de recuperación del capital.

Para llevar a cabo un diagnóstico exhaustivo e imparcial, este apartado articula el análisis *out-of-sample* (OOS) en torno a tres dimensiones analíticas complementarias:

* **Métricas de rentabilidad y volatilidad:** Miden la magnitud del crecimiento patrimonial y el nivel de dispersión diario.
    * **Cumulative Return:** Rentabilidad acumulada total a lo largo del horizonte *out-of-sample*.
    * **CAGR (*Compound Annual Growth Rate*):** Tasa de crecimiento anual compuesta que normaliza el rendimiento temporal.
    * **Annualized Volatility:** Volatilidad de los retornos diarios multiplicada por el factor de anualización ($\sqrt{252}$).


* **Métricas de rendimiento ajustado al riesgo:** Evalúan la eficiencia en la conversión de riesgo en retorno.
    * **Sharpe Ratio:** Exceso de rentabilidad por unidad de volatilidad total.
    * **Sortino Ratio:** Exceso de retorno por unidad de volatilidad a la baja (*downside risk*), penalizando únicamente los retornos negativos.
    * **Calmar Ratio:** Relación entre el CAGR y el *Maximum Drawdown*, que mide el retorno obtenido por cada unidad de pérdida máxima histórica.


* **Métricas de *drawdown* y comportamiento en momentos de estrés:** Analizan la resiliencia del capital ante fases adversas de mercado.
    * **Maximum Drawdown (MDD):** Mayor pérdida porcentual acumulada desde un máximo histórico previo (*peak-to-trough*).
    * **Average Drawdown:** Pérdida media observada a lo largo de todos los episodios de *drawdown*, evitando que la evaluación de riesgo dependa exclusivamente de un único evento extremo puntual.
    * **Underwater Duration (o Duration in Drawdown)**: Número máximo de sesiones de negociación consecutivas en las que el capital se encuentra por debajo de su máximo histórico anterior.

Este marco multidimensional permite determinar qué arquitecturas de predicción y esquemas de ponderación no solo generan alfa bruto, sino que logran consolidar carteras sólidas, eficientes e inmunes a la erosión operativa en el entorno real.


### 3.1 Overall Net Performance

In [5]:
from src.portfolio.metrics import calculate_performance_metrics

performance_metrics = calculate_performance_metrics(net_returns)

3.1 — NET PORTFOLIO PERFORMANCE
✓ Strategies evaluated = 42
✓ Models evaluated = 3
✓ Date range = 2025-01-16 → 2026-08-10
✓ Missing performance metrics = 0

        model                           portfolio   CAGR Ann_vol Sharpe Sortino Calmar  max_DD  max_underwater_in_days
        Ridge      long_only_top_10_signal_weight 51.21%  26.26%  1.950   2.952  1.900 -26.95%                     108
        Ridge              long_only_equal_weight 47.59%  25.42%  1.872   2.847  1.791 -26.57%                     109
Random Forest      long_only_top_10_signal_weight 66.94%  36.73%  1.823   2.688  2.054 -32.59%                      86
        Ridge      long_only_top_20_signal_weight 45.65%  25.13%  1.817   2.774  1.717 -26.59%                     108
      XGBoost      long_only_top_20_signal_weight 55.63%  30.83%  1.804   2.706  1.904 -29.22%                      81
      XGBoost      long_only_top_30_signal_weight 48.01%  27.51%  1.745   2.644  1.824 -26.32%                      76
      XGBo

El análisis comparativo de las estrategias evaluadas revela diferencias sustanciales en el rendimiento ajustado por riesgo y la magnitud del drawdown, en función del modelo de aprendizaje automático y del esquema de construcción de cartera utilizados.

En términos de eficiencia ajustada por riesgo, la combinación de Ridge con la cartera `long_only_top_10_signal_weight` alcanza el Sharpe Ratio más elevado de la muestra ($1.950$), secundada por la variante con ponderación equitativa (`long_only_equal_weight`, $1.872$) sobre el mismo modelo. Esta configuración de Ridge (`long_only_top_10_signal_weight`) también registra la mejor asimetría en la rentabilidad ajustada a la baja, con un Sortino Ratio de $2.952$.

Desde la perspectiva de la rentabilidad absoluta, los modelos no lineales lideran la comparativa: Random Forest junto con `long_only_top_10_signal_weight` genera el mayor CAGR del estudio ($66.94\%$), seguido de cerca por la variante equivalente en XGBoost ($62.08\%$). No obstante, esta mayor rentabilidad viene acompañada de una mayor volatilidad anualizada ($36.73\%$ y $35.60\%$, respectivamente) y de caídas máximas (*maximum drawdown*) más pronunciadas, que superan el $-32\%$ ($-32.59\%$ y $-32.49\%$).

El control del riesgo destaca en las estrategias Long-Short y en las configuraciones de Paridad de Riesgo de 20 activos. Las carteras `long_short_equal_weight` exhiben los drawdowns máximos más acotados de la muestra ($-12.25\%$ para Ridge y $-13.23\%$ para XGBoost), manteniendo volatilidades anualizadas contenidas entre el $11.81\%$ y el $15.07\%$. Por su parte, la combinación de Random Forest con `long_short_equal_weight` logra el periodo de permanencia bajo máximos (*maximum underwater duration*) más breve, con solo $75$ días de negociación y un drawdown máximo contenido en el $-13.28\%$.

En el extremo opuesto, las carteras de Inverse Volatility de 30 activos Random Forest y XGBoost se sitúan en el tramo inferior de la tabla, cerrando la clasificación con ratios de Sharpe entre $0.811$ y $0.799$, junto con reducciones sustanciales en la eficiencia del ratio Calmar.

### 3.2 Agregated Performance Analysis

Para sintetizar los resultados, las métricas se agrupan promediando el rendimiento de las carteras en tres dimensiones analíticas: el **modelo** de aprendizaje automático subyacente (*Ridge*, *XGBoost* y *Random Forest*), la **filosofía** de construcción de cartera utilizada, y el **tamaño del universo** de activos seleccionados (*Top 10*, *Top 20* y *Top 30*).

In [6]:
# =============================================================================
# Performance Analysis by Model, Portfolio Type and Universe
# =============================================================================

# =============================================================================
# 1. Performance by ML Model
# =============================================================================

model_comparison = (
    performance_metrics
    .groupby("model")[
        [
            "CAGR",
            "Ann_vol",
            "Sharpe",
            "Sortino",
            "Calmar",
            "max_DD",
            "max_underwater_in_days",
        ]
    ]
    .mean()
    .sort_values("Sharpe", ascending=False)
)

print("=" * 80)
print("3.2 — PERFORMANCE BY ML MODEL")
print("=" * 80)

print(
    model_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "Ann_vol": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
            "max_underwater_in_days": lambda x: f"{x:.0f}",
        }
    )
)


# =============================================================================
# 2. Performance by Portfolio Philosophy
# =============================================================================

from src.portfolio.metrics import extract_weighting_scheme


performance_metrics["weighting_scheme"] = (
    performance_metrics["portfolio"]
    .apply(extract_weighting_scheme)
)

portfolio_type_comparison = (
    performance_metrics
    .groupby("weighting_scheme")[
        [
            "CAGR",
            "Ann_vol",
            "Sharpe",
            "Sortino",
            "Calmar",
            "max_DD",
            "max_underwater_in_days",
        ]
    ]
    .mean()
    .sort_values("Sharpe", ascending=False)
)

print("\n" + "=" * 80)
print("3.2 — PERFORMANCE BY PORTFOLIO PHILOSOPHY")
print("=" * 80)

print(
    portfolio_type_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "Ann_vol": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "max_DD": lambda x: f"{x:.2%}",
            "max_underwater_in_days": lambda x: f"{x:.0f}",
        }
    )
)


# =============================================================================
# 3. Performance by Portfolio Universe
# =============================================================================

from src.portfolio.metrics import extract_universe

performance_metrics["universe"] = (
    performance_metrics["portfolio"]
    .apply(extract_universe)
)

universe_comparison = (
    performance_metrics
    .groupby("universe")[
        [
            "CAGR",
            "Ann_vol",
            "Sharpe",
            "Sortino",
            "Calmar",
            "max_DD",
            "max_underwater_in_days",
        ]
    ]
    .mean()
)


universe_order = [
    "Full / Baseline",
    "Top 10%",
    "Top 20%",
    "Top 30%",
]

universe_comparison = (
    universe_comparison
    .reindex(
        [x for x in universe_order if x in universe_comparison.index]
    )
)

print("\n" + "=" * 80)
print("3.2 — PERFORMANCE BY PORTFOLIO UNIVERSE")
print("=" * 80)

print(
    universe_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "Ann_vol": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "max_DD": lambda x: f"{x:.2%}",
            "max_underwater_in_days": lambda x: f"{x:.0f}",
        }
    )
)

3.2 — PERFORMANCE BY ML MODEL
                CAGR Ann_vol Sharpe Sortino Calmar    max_DD max_underwater_in_days
model                                                                              
Ridge         33.41%  21.64%  1.523   2.287  1.445 -0.229062                    103
XGBoost       36.69%  23.69%  1.496   2.222  1.517 -0.234356                     95
Random Forest 36.34%  23.92%  1.470   2.185  1.530 -0.230627                     90

3.2 — PERFORMANCE BY PORTFOLIO PHILOSOPHY
                     CAGR Ann_vol Sharpe Sortino Calmar  max_DD max_underwater_in_days
weighting_scheme                                                                      
Other              52.08%  29.47%  1.765   2.655  1.824 -28.43%                     90
Equal Weight       36.98%  22.23%  1.617   2.379  1.707 -21.20%                     92
Maximum Sharpe     31.18%  21.26%  1.467   2.153  1.421 -21.97%                     92
Risk Parity        29.54%  20.43%  1.428   2.139  1.362 -21.49%         

Al agregar los datos por modelo de Machine Learning, XGBoost lidera el retorno anualizado ($36.69\%$), seguido muy de cerca por Random Forest ($36.34\%$). Sin embargo, Ridge obtiene el mayor ratio Sharpe ($1.523$) y Sortino ($2.287$), impulsado por la volatilidad anualizada más reducida del grupo ($21.64\%$). Los tres modelos presentan un perfil de riesgo ajustado muy similar, con drawdowns máximos contenidos entre el $-22.91\%$ y el $-23.44\%$, siendo Random Forest el que muestra la recuperación más rápida tras caídas ($90$ días underwater).

En cuanto a la filosofía de construcción de cartera, la categoría Other (que engloba las estrategias Signal Weight y Long-Short) destaca holgadamente con el mayor CAGR ($52.08\%$), el Sharpe más elevado ($1.765$) y el mayor Sortino ($2.655$), a costa de asumir mayor volatilidad ($29.47\%$) y un drawdown de $-28.43\%$. Entre las filosofías tradicionales, Equal Weight ofrece la mejor eficiencia global (Sharpe de $1.617$ y Calmar de $1.707$) con un drawdown de $-21.20\%$. Por su parte, Maximum Sharpe, Risk Parity e Inverse Volatility logran acotar la volatilidad anualizada ($20.43\% \text{ – } 21.74\%$) y el drawdown máximo alrededor del $-21\% \text{ – } -22\%$, aunque con un CAGR más modesto ($28.12\% \text{ – } 31.18\%$).

Por último, el análisis según el tamaño del universo confirma una clara relación entre concentración y rentabilidad: restringir la muestra al Top 10% maximiza el CAGR ($44.64\%$) y el ratio Sharpe ($1.632$), pero asume la mayor volatilidad ($27.15\%$) y el drawdown más severo ($-27.20\%$). A medida que el universo se amplía hacia el Top 20% y Top 30%, la volatilidad disminuye hasta el $20.31\%$ y la caída máxima al $-20.76\%$, acompañados de una modulación lógica en la rentabilidad anualizada ($33.07\%$ y $27.97\%$, respectivamente). El universo Full / Baseline mantiene un desempeño sólido impulsado por sus métricas de Equal Weight, con un CAGR del $36.98\%$ y el tiempo en recuperación más bajo ($92$ días).

## 4. Performance Interpretation

El comportamiento divergente observado entre los modelos, las metodologías de construcción de cartera y el tamaño de los universos responde a mecánicas económico-financieras y cuantitativas bien documentadas.

### 4.1 Efecto de Regularización vs. Captura de No Linealidades

La superioridad de *Ridge* en los ratios de eficiencia (*Sharpe* y *Sortino* en cartera individual) radica en su penalización L2, la cual reduce la varianza de los coeficientes frente a la colinealidad de los *factors* de mercado. Al emitir estimaciones de retorno más conservadoras y estables, minimiza las rotaciones innecesarias y el ruido de asignación. Por el contrario, *XGBoost* y *Random Forest* destacan en retorno absoluto al mapear interacciones no lineales complejas entre *features*; sin embargo, esta agresividad predictiva induce una mayor volatilidad estructural en las ponderaciones, aumentando las caídas temporales.

### 4.2 El Ratio de Información de la Señal (Signal Weighting)

El elevado desempeño del esquema *Signal Weight* confirma la existencia de un alpha monótono en las predicciones: la magnitud del valor predicho por los modelos no solo indica la dirección del activo, sino también la convicción de la anomalía. Ponderar en función de la intensidad de la señal maximiza la captura de este alpha en comparación con asignaciones pasivas (*Equal Weight*), compensando el incremento asumido en la volatilidad.

### 4.3 Fallo out-of-sample de la Optimización Media-Varianza

El modesto rendimiento relativo de las carteras basadas en Maximum Sharpe (Sharpe medio de $1.388$ frente al $1.614$ de Equal Weight) evidencia la vulnerabilidad clásica de la optimización de varianza media frente a los errores de estimación out-of-sample (Michaud, 1989). Al maximizar el ratio en la ventana de entrenamiento, el algoritmo asigna pesos extremos a los activos donde las estimaciones de retorno esperado o covarianza contienen mayor ruido. Al evaluar la estrategia fuera de la muestra, la inestabilidad de estos pesos y el coste implícito de su rebalanceo penalizan de manera severa el perfil ajustado por riesgo en comparación con heurísticas de ponderación más robustas.

### 4.4 Trade-off de Concentración y Diversificación

La degradación progresiva del *CAGR* al pasar del *Top 10%* al *Top 30%* evidencia que la capacidad predictiva del modelo se concentra en los extremos de la distribución (*tail alpha*). Incorporar un mayor número de activos diluye la prima de riesgo capturada por la señal ML a cambio de reducir la volatilidad total, confirmando que el poder discriminatorio del modelo decrece rápidamente a medida que se desciende en el ranking de predicción.

### 4.5 Simetría y Cobertura en Carteras Long-Short

La contención en las métricas de *drawdown* de las carteras *Long-Short* responde a la neutralización parcial del riesgo sistemático (*Beta*). Al tomar posiciones cortas en los activos con peores señales predichas, la estrategia aísla el rendimiento idiosincrásico de la selección de activos, sacrificando la prima de riesgo del mercado a cambio de una trayectoria de capital sustancialmente más estable durante episodios de estrés bursátil.

### 4.6 Rentabilidad Bruta frente a Rentabilidad Neta

La comparación entre gross performance y net performance constituye un filtro adicional de robustez. Una estrategia que presenta un elevado CAGR bruto pero requiere una rotación excesiva puede perder buena parte de su ventaja una vez incorporados los costes de transacción. Por tanto, la rentabilidad neta debe considerarse la métrica económicamente relevante para evaluar la viabilidad de implementación. Este efecto es especialmente importante en Maximum Sharpe, donde la elevada sensibilidad de los pesos genera una penalización operativa sustancial.

### 4.7 Rentabilidad Ajustada por Riesgo frente a Rentabilidad Absoluta

Finalmente, el CAGR no debe interpretarse de forma aislada. El Sharpe y el Sortino permiten evaluar cuánto retorno se obtiene por unidad de riesgo, mientras que el Calmar relaciona la rentabilidad anualizada con el Maximum Drawdown. La duración del período underwater añade una dimensión temporal que permite distinguir entre una caída profunda pero rápidamente recuperada y una estrategia que permanece durante largos períodos por debajo de sus máximos históricos. Por ello, la selección final de estrategias debe basarse en el conjunto de métricas y no exclusivamente en la rentabilidad acumulada.

## 5. Rebalancing Frequency Sensitivity Analysis


### 5.1 Sensitivity Framework & Frequency Grid

La selección de la frecuencia de rebalanceo representa uno me los compromisos (*trade-offs*) más críticos en la gestión de carteras cuantitativas. Mientras que una reponderación muy frecuente permite adaptar rápidamente las posiciones a la evolución de las señales predictivas, también dispara la rotación de activos (*turnover*) y la acumulación de costes de transacción. Por el contrario, frecuencias demasiado dilatadas reducen la fricción operativa a costa de permitir un desalineamiento progresivo de la cartera respecto a las ponderaciones óptimas objetivo.

Entre fechas de rebalanceo consecutivas, la cartera opera bajo una dinámica *buy-and-hold*. Durante este intervalo, la rentabilidad dispar de los componentes provoca una **deriva de los pesos (*weight drift*)**, donde los activos ganadores ganan peso relativo en la estrategia y los perdedores pierden peso. Este fenómeno genera dos efectos contrapuestos:

1. **Deterioro de la cartera objetivo:** El desajuste por *drift* puede desviar el perfil de riesgo-retorno y diluir la capacidad del modelo cuantitativo para capturar las ineficiencias de mercado detectadas en $T$.
2. **Medición del *turnover* real:** La rotación en el evento de rebalanceo $T+1$ no se calcula con respecto a los pesos nominales asignados en $T$, sino con respecto a los pesos efectivamente desviados (*drifted weights*) justo antes de ejecutar las nuevas órdenes. Ignorar este ajuste distorsiona tanto la simulación de retornos intermedios como la cuantificación exacta de los costes de transacción.

Para evaluar este impacto y determinar la ventana temporal que maximiza la eficiencia neta, este apartado extiende el universo analítico evaluando la sensibilidad de las estrategias ante cinco horizontes de rebalanceo alternativos: **5 días** (semanal), **10 días** (bisemanal), **21 días** (mensual, escenario base), **42 días** (bimensual) y **63 días** (trimestral).

La comparación entre horizontes se estructura analizando de forma simultánea la mecánica operativa con *drift* diario y el rendimiento final a través del siguiente bloque de métricas:

* **Dinámica operativa y fricción:** Se cuantifica la tasa de rotación anualizada (*Turnover* ajustado por *drift*) y la erosión por costes de ejecución (*Transaction Costs*), aislando el desgaste generado por cada frecuencia.
* **Preservación de retorno y riesgo:** Se evalúa la evolución de la rentabilidad bruta frente a la neta mediante el **CAGR neto** —establecido como la métrica central de decisión— y la rentabilidad acumulada (*Cumulative Net Return*).
* **Eficiencia y resiliencia:** Se mide la conservación del perfil de riesgo-retorno ajustado a través del *Sharpe Ratio*, *Sortino Ratio*, la profundidad de las caídas (*Maximum Drawdown*) y el tiempo de recuperación del capital (*Maximum Underwater Duration*).

El objetivo de este marco es identificar el punto de equilibrio óptimo donde la frescura de la señal predictiva compense con creces la fricción de negociación y el efecto del *drift*, fundamentando empíricamente la elección del calendario de rebalanceo para su despliegue en entornos reales de inversión.

### 5.2 Frequency Sensitivity — Execution Engine

In [7]:
# =============================================================================
# Rebalancing Frequency Sensitivity
# =============================================================================

REBALANCING_FREQUENCIES = [5, 10, 21, 42, 63]

# Load final portfolio weights and extended S&P 500 asset prices
final_portfolio_weights = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

prices = pd.read_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)

final_portfolio_weights["date"] = pd.to_datetime(
    final_portfolio_weights["date"]
)

# =============================================================================
# Compute Daily Asset Returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

simple_returns.index = pd.to_datetime(
    simple_returns.index
)

all_trading_dates = pd.Index(
    sorted(simple_returns.index.unique())
)


from src.portfolio.sensitivity import run_frequency_sensitivity_execution

frequency_gross_returns, frequency_turnover = (
    run_frequency_sensitivity_execution(
        weights_raw=final_portfolio_weights,
        asset_returns=simple_returns,
        all_trading_dates=all_trading_dates,
        rebalancing_frequencies=REBALANCING_FREQUENCIES,
        verbose=True,
    )
)

REBALANCING FREQUENCY SENSITIVITY WITH DRIFT — EXECUTION AUDIT
✓ Frequencies tested = [5, 10, 21, 42, 63]
✓ Total daily observations = 73,416
✓ Unique frequencies = 5
✓ Unique portfolios = 14
✓ Unique models = 3
✓ Total portfolios evaluated = 210
✓ Date range = 2025-01-16 → 2026-07-09
✓ Missing gross returns = 0
✓ Duplicate observations = 0


### 5.3 Trade-off Analysis: Turnover vs. Net CAGR

In [8]:
# =============================================================================
# Configuration
# =============================================================================

REBALANCING_FREQUENCIES = {
    5: "Weekly",
    10: "Biweekly",
    21: "Monthly",
    42: "Bimonthly",
    63: "Quarterly",
}

BASE_TRANSACTION_COST = 0.0015       # 15 bps
TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0

# =============================================================================
# Frequency Sensitivity Engine
# =============================================================================

from src.portfolio.sensitivity import evaluate_frequency_tradeoff

frequency_sensitivity = evaluate_frequency_tradeoff(
       frequency_gross_returns=frequency_gross_returns,
       frequency_turnover=frequency_turnover,
       rebalancing_frequencies=REBALANCING_FREQUENCIES,
       base_transaction_cost=BASE_TRANSACTION_COST,  # 15 bps
       verbose=True,
   )


REBALANCING FREQUENCY TRADEOFF ANALYSIS — AUDIT
✓ Frequencies evaluated = [5, 10, 21, 42, 63]
✓ Applied transaction cost = 15.0 bps
✓ Total strategy-frequency observations = 210
✓ Missing net CAGR = 0
✓ Missing net Sharpe = 0


In [9]:
# =============================================================================
# Table 1 — Frequency Sensitivity: Core Economic Metrics (Mean & Median)
# =============================================================================

frequency_core = (
    frequency_sensitivity.groupby(["frequency_days", "frequency_label"])
    .agg(
        mean_CAGR=("CAGR", "mean"),
        median_CAGR=("CAGR", "median"),
        mean_turnover=("annualized_turnover", "mean"),
        median_turnover=("annualized_turnover", "median"),
        mean_tc=("cumulative_transaction_cost", "mean"),
        mean_net_return=("cumulative_net_return", "mean"),
    )
    .reset_index()
)

print("=" * 80)
print("TABLE 1 — REBALANCING FREQUENCY: ECONOMIC TRADE-OFF")
print("=" * 80)

print(
    frequency_core.to_string(
        index=False,
        formatters={
            "mean_CAGR": lambda x: f"{x:.2%}",
            "median_CAGR": lambda x: f"{x:.2%}",
            "mean_turnover": lambda x: f"{x:.2%}",
            "median_turnover": lambda x: f"{x:.2%}",
            "mean_tc": lambda x: f"{x:.2%}",    
            "mean_net_return": lambda x: f"{x:.2%}",
        },
    )
)

TABLE 1 — REBALANCING FREQUENCY: ECONOMIC TRADE-OFF
 frequency_days frequency_label mean_CAGR median_CAGR mean_turnover median_turnover mean_tc mean_net_return
              5          Weekly    37.11%      33.08%       326.43%         343.51%   0.72%          59.68%
             10        Biweekly    36.74%      33.28%       288.97%         304.99%   0.63%          59.03%
             21         Monthly    38.74%      32.77%       258.04%         268.45%   0.51%          59.90%
             42       Bimonthly    33.71%      33.05%       176.28%         181.53%   0.31%          47.72%
             63       Quarterly    35.37%      32.44%       136.99%         142.28%   0.20%          46.34%


El análisis con deriva de peso (*weight drift*) confirma una reducción monótona de la fricción operativa al dilatar el horizonte de rebalanceo: la rotación media anualizada (*turnover*) se reduce del **326.43%** en 5 días al **136.99%** en 63 días, disminuyendo el coste de transacción medio del **0.72%** al **0.20%**.

Sin embargo, la frescura de la señal predictiva domina sobre la fricción operativa hasta el horizonte mensual. El rendimiento neto alcanza su máximo global en la frecuencia **mensual (21 días)**, registrando un **38.74% de CAGR medio** y un **59.90% de retorno neto acumulado**.

* **Frecuencias de ultra corto plazo (5–10 días):** Generan retornos netos sólidos (~37% CAGR medio), pero asumen un coste y un *turnover* tres veces superior sin aportar *alpha* adicional.
* **Frecuencias dilatadas (42–63 días):** La reducción de costes no compensa la degradación de la señal (*signal decay*) ni el *drift* acumulado, provocando una caída severa del retorno neto acumulado medio hasta el **47.72%** (42 días) y **46.34%** (63 días).

In [10]:
# =============================================================================
# Table 2 — Frequency Sensitivity: Risk-Adjusted Performance
# =============================================================================

frequency_risk = (
    frequency_sensitivity.groupby(["frequency_days", "frequency_label"])
    .agg(
        median_Sharpe=("Sharpe", "median"),
        mean_Sharpe=("Sharpe", "mean"),
        median_Sortino=("Sortino", "median"),
        mean_MaxDD=("maximum_drawdown", "mean"),
        median_Underwater_Days=(
            "maximum_underwater_duration_days",
            "median",
        ),
    )
    .reset_index()
)

print("=" * 80)
print("TABLE 2 — REBALANCING FREQUENCY: RISK-ADJUSTED PERFORMANCE")
print("=" * 80)

print(
    frequency_risk.to_string(
        index=False,
        formatters={
            "median_Sharpe": lambda x: f"{x:.3f}",
            "mean_Sharpe": lambda x: f"{x:.3f}",
            "median_Sortino": lambda x: f"{x:.3f}",
            "mean_MaxDD": lambda x: f"{x:.2%}",
            "median_Underwater_Days": lambda x: f"{x:.0f}",
        },
    )
)

TABLE 2 — REBALANCING FREQUENCY: RISK-ADJUSTED PERFORMANCE
 frequency_days frequency_label median_Sharpe mean_Sharpe median_Sortino mean_MaxDD median_Underwater_Days
              5          Weekly         1.531       1.533          2.283    -23.20%                     98
             10        Biweekly         1.561       1.522          2.298    -23.74%                    104
             21         Monthly         1.530       1.584          2.262    -22.98%                     99
             42       Bimonthly         1.495       1.380          2.174    -23.99%                    108
             63       Quarterly         1.489       1.413          2.197    -23.80%                    102


El comportamiento del riesgo a lo largo del abanico de frecuencias ratifica al horizonte mensual (21 días) como la opción más robusta, registrando el Sharpe medio más alto (1.584) del experimento.

La media del ratio de Sharpe se mantiene elevada entre los 5 y 21 días (~1.52 a 1.58), pero sufre un deterioro claro al dilatar el rebalanceo a horizontes bimestrales (1.380 en 42 días) y trimestrales (1.413 en 63 días). La mediana del ratio de Sortino sigue la misma tendencia, cayendo de 2.298 (bisemanal) y 2.262 (mensual) hasta 2.174 (bimestral). Por otro lado, el Maximum Drawdown medio permanece contenido en una franja estrecha entre el -22.98% (mensual) y el -23.99% (bimestral). Esto confirma que las caídas de cola son una propiedad estructural de la selección de activos y no del calendario de ejecución.

In [11]:
# =============================================================================
# Table 3 — Best Risk-Adjusted Strategies Across All Frequencies
# =============================================================================

best_overall = frequency_sensitivity.sort_values(
    "Sharpe", ascending=False
).head(20)

print("=" * 80)
print("TABLE 3 — TOP 20 STRATEGIES ACROSS ALL FREQUENCIES (BY SHARPE)")
print("=" * 80)

print(
    best_overall[
        [
            "frequency_label",
            "model",
            "portfolio",
            "CAGR",
            "Sharpe",
            "Sortino",
            "annualized_turnover",
            "maximum_drawdown",
        ]
    ].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "annualized_turnover": lambda x: f"{x:.2%}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

TABLE 3 — TOP 20 STRATEGIES ACROSS ALL FREQUENCIES (BY SHARPE)
frequency_label         model                           portfolio   CAGR Sharpe Sortino annualized_turnover maximum_drawdown
        Monthly Random Forest      long_only_top_10_signal_weight 89.75%  2.474   3.663             219.69%          -32.47%
        Monthly       XGBoost      long_only_top_10_signal_weight 81.63%  2.309   3.412             234.84%          -32.37%
        Monthly       XGBoost             long_short_equal_weight 32.45%  2.232   3.238             199.28%          -12.85%
        Monthly Random Forest              long_only_equal_weight 72.42%  2.205   3.279             233.38%          -30.45%
        Monthly       XGBoost      long_only_top_20_signal_weight 68.14%  2.198   3.293             227.14%          -29.09%
        Monthly Random Forest             long_short_equal_weight 33.09%  2.178   3.159             254.86%          -12.35%
        Monthly       XGBoost              long_only_equal_wei

El análisis cruzado de las veinte mejores configuraciones ajustadas por riesgo confirma la clara hegemonía de la frecuencia mensual (21 días), que acapara de forma consecutiva las primeras ocho posiciones del ranking global. La combinación más eficiente del experimento la encabeza Random Forest bajo la arquitectura long_only_top_10_signal_weight, registrando un Sharpe de 2.474, un CAGR neta del 89.75% y un Sortino de 3.663 con una rotación anualizada del 219.69%.

A nivel metodológico, la asignación ponderada por la intensidad de la señal (signal weight) sobre el Top 10 demuestra ser la estrategia más alpha-generativa de forma transversal en todos los algoritmos (Random Forest, XGBoost y Ridge). Por su parte, las estrategias no lineales evidencian una notable capacidad para preservar la rentabilidad en horizontes dilatados: en la frecuencia trimestral (63 días), tanto Random Forest como XGBoost sostienen ratios de Sharpe superiores a 2.0 y CAGRs por encima del 72%, reduciendo el turnover a un eficiente ~119%. Finalmente, la estructura long_short_equal_weight a 21 días destaca como la alternativa más resiliente al riesgo, limitando el drawdown máximo al entorno del -12% y conservando un Sharpe superior a 2.17.

### 5.4. Full Cross-Sectional Experimentation Grid

Con el fin de ofrecer una visión exhaustiva del espacio de búsqueda, esta sección consolida la totalidad de las iteraciones ejecutadas en el marco experimental. La tabla interactiva expuesta a continuación integra las dimensiones analizadas en las secciones previas: los tres modelos de aprendizaje automático (Ridge, Random Forest y XGBoost), los catorce esquemas de construcción de cartera y las cinco frecuencias de rebalanceo ($5$, $10$, $21$, $42$ y $63$ días). 

Esta matriz multidimensional permite examinar de manera granular las interacciones entre el algoritmo subyacente, la asignación de pesos y la velocidad de rotación, facilitando la identificación de combinaciones óptimas que maximizan el retorno ajustado por riesgo al tiempo que contienen las fricciones operativas.

In [12]:
# =============================================================================
# Table — Performance & Rebalancing Sensitivity
# =============================================================================

# -------------------------------------------------------------------------
# Define ordering
# -------------------------------------------------------------------------

FREQUENCY_ORDER = {
    5: 0,
    10: 1,
    21: 2,
    42: 3,
    63: 4,
}

MODEL_ORDER = {
    "Ridge": 0,
    "Random Forest": 1,
    "XGBoost": 2,
}


# =============================================================================
# Prepare Table & Calculate Calmar Ratio if not present
# =============================================================================

performance_frequency_table = frequency_sensitivity.copy()

if "Calmar" not in performance_frequency_table.columns:
    performance_frequency_table["Calmar"] = np.where(
        performance_frequency_table["maximum_drawdown"] < 0,
        performance_frequency_table["CAGR"]
        / performance_frequency_table["maximum_drawdown"].abs(),
        np.nan,
    )

# -------------------------------------------------------------------------
# Create temporary sorting keys
# -------------------------------------------------------------------------

performance_frequency_table["_frequency_order"] = (
    performance_frequency_table["frequency_days"].map(FREQUENCY_ORDER)
)

performance_frequency_table["_model_order"] = (
    performance_frequency_table["model"].map(MODEL_ORDER)
)


# =============================================================================
# Sort
# =============================================================================

performance_frequency_table = (
    performance_frequency_table.sort_values(
        [
            "_frequency_order",
            "_model_order",
            "portfolio",
        ]
    )
    .drop(
        columns=[
            "_frequency_order",
            "_model_order",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# Validation
# =============================================================================

assert set(performance_frequency_table["frequency_days"]) == {
    5,
    10,
    21,
    42,
    63,
}

assert (
    performance_frequency_table[
        [
            "CAGR",
            "annualized_volatility",
            "annualized_turnover",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
        ]
    ]
    .notna()
    .all()
    .all()
)


# =============================================================================
# Audit Output
# =============================================================================

print("=" * 110)
print(
    "PERFORMANCE & REBALANCING SENSITIVITY — "
    "NET RETURNS (BASE SCENARIO — 15 BPS)"
)
print("=" * 110)

print(
    f"✓ Frequencies evaluated = "
    f"{performance_frequency_table['frequency_days'].nunique()}"
)

print(
    f"✓ Total strategy-frequency combinations = "
    f"{len(performance_frequency_table):,}"
)

print()

PERFORMANCE & REBALANCING SENSITIVITY — NET RETURNS (BASE SCENARIO — 15 BPS)
✓ Frequencies evaluated = 5
✓ Total strategy-frequency combinations = 210



In [13]:
performance_frequency_table.to_parquet(
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet",
    index=False,
)


## 6. Subperiod Performance & Temporal Stability Analysis

Para validar la solidez de los hallazgos no basta con evaluar el rendimiento medio global a lo largo de todo el horizonte *out-of-sample* (OOS). Un retorno acumulado sobresaliente puede ser el resultado engañoso de un único periodo excepcionalmente favorable, ocultando episodios prolongados de mediocridad o volatilidad desmedida. Para responder a una cuestión clave —si la superioridad de una estrategia es estructural o meramente coyuntural—, este apartado introduce una segunda dimensión analítica basada en el **análisis por subperiodos y la consistencia temporal**.

El periodo OOS total se divide en tres fases temporales diferenciadas de similar duración (*Early OOS*, *Middle OOS* y *Late OOS*). Para cada una de estas ventanas y cada combinación de modelo y esquema de ponderación, se replican las métricas de rendimiento neto y comportamiento operativo fundamentales:

$$\text{Métricas por subperiodo} = \{\text{CAGR}, \text{Sharpe Ratio}, \text{Sortino Ratio}, \text{Maximum Drawdown}, \text{Recovery Time}, \text{Turnover}\}$$

**Evaluación de la Consistencia Temporal**

Junto al desglose por fases, se incorpora una métrica cuantitativa directa de persistencia: la **Tasa de Consistencia de Batiendo al Benchmark ($\text{Consistency } \%$)**. Esta variable mide el porcentaje de subperiodos en los que una estrategia dada supera en rendimiento neto ($R^{\text{net}}$) a la referencia pasiva de referencia (*Equal Weight* o referencia del mercado):

$$\text{Consistency } \% = \frac{\sum_{p=1}^{P} \mathbb{I}\left(\text{CAGR}_{\text{Estrategia}, p} > \text{CAGR}_{\text{Benchmark}, p}\right)}{P} \times 100$$

donde $P$ representa el número total de subperiodos evaluados y $\mathbb{I}(\cdot)$ es la función indicadora.

Este enfoque permite discriminar entre arquitecturas que ofrecen alfa genuino y persistente en distintas condiciones de mercado de aquellas dependientes de regímenes específicos, consolidando un marco de comparación estricto para la selección final de modelos.


### 6.1 Subperiod Performance Evaluation (Early, Middle, Late OOS)

In [14]:
# =============================================================================
# 6.1 Subperiod Performance Evaluation (Early, Middle, Late OOS)
# =============================================================================

from src.portfolio.subperiods import compute_subperiod_performance_with_drift

# -------------------------------------------------------------------------
# Load Datasets
# -------------------------------------------------------------------------
net_returns = pd.read_parquet(
    "../data/portfolio_results/net_portfolio_returns.parquet"
)
weights_raw = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

# -------------------------------------------------------------------------
# Run Computation with Drift Integration
# -------------------------------------------------------------------------
subperiod_performance, date_period_map = (
    compute_subperiod_performance_with_drift(
        net_returns=net_returns,
        weights_df=weights_raw,
        simple_returns=simple_returns,
        rebalancing_days=21,
    )
)

# -------------------------------------------------------------------------
# Validation Assertions
# -------------------------------------------------------------------------
expected_periods = {"Early OOS", "Middle OOS", "Late OOS"}
assert set(subperiod_performance["period"]) == expected_periods
assert (
    subperiod_performance[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "maximum_drawdown",
            "annualized_turnover",
        ]
    ]
    .notna()
    .all()
    .all()
)

# -------------------------------------------------------------------------
# Audit Output
# -------------------------------------------------------------------------
period_counts = date_period_map["period"].value_counts().reindex(expected_periods)

print("=" * 80)
print("6.1 — SUBPERIOD PERFORMANCE ANALYSIS (WITH DRIFT)")
print("=" * 80)

print(f"✓ Total OOS dates = {len(date_period_map):,}")
print(f"✓ Early OOS dates = {period_counts['Early OOS']:,}")
print(f"✓ Middle OOS dates = {period_counts['Middle OOS']:,}")
print(f"✓ Late OOS dates = {period_counts['Late OOS']:,}")
print(f"✓ Strategies evaluated = {len(subperiod_performance):,}")
print(f"✓ Models evaluated = {subperiod_performance['model'].nunique():,}")
print(f"✓ Missing metrics = {subperiod_performance.isna().sum().sum():,}")

print("\nPERIOD DATE RANGES")
print("-" * 80)

for period in ["Early OOS", "Middle OOS", "Late OOS"]:
    period_dates = date_period_map[date_period_map["period"] == period]["date"]
    print(
        f"{period:<12} {period_dates.min().date()} → {period_dates.max().date()}"
    )

print("=" * 80)

6.1 — SUBPERIOD PERFORMANCE ANALYSIS (WITH DRIFT)
✓ Total OOS dates = 392
✓ Early OOS dates = 130
✓ Middle OOS dates = 130
✓ Late OOS dates = 132
✓ Strategies evaluated = 126
✓ Models evaluated = 3
✓ Missing metrics = 0

PERIOD DATE RANGES
--------------------------------------------------------------------------------
Early OOS    2025-01-16 → 2025-07-24
Middle OOS   2025-07-25 → 2026-01-29
Late OOS     2026-01-30 → 2026-08-10


### 6.2 Model Stability across Market Regimes

In [15]:
# -------------------------------------------------------------------------
# Common Setup & Formatters
# -------------------------------------------------------------------------
metrics_to_aggregate = [
    "CAGR",
    "annualized_volatility",
    "Sharpe",
    "Sortino",
    "maximum_drawdown",
    "annualized_turnover",
]

formatters = {
    "CAGR": lambda x: f"{x:.2%}",
    "annualized_volatility": lambda x: f"{x:.2%}",
    "Sharpe": lambda x: f"{x:.3f}",
    "Sortino": lambda x: f"{x:.3f}",
    "maximum_drawdown": lambda x: f"{x:.2%}",
    "annualized_turnover": lambda x: f"{x:.2%}",
}

period_order = ["Early OOS", "Middle OOS", "Late OOS"]

# Ensure period categorical ordering
subperiod_performance["period"] = pd.Categorical(
    subperiod_performance["period"],
    categories=period_order,
    ordered=True,
)

In [16]:
# =============================================================================
# Table 1: Performance Grouped by Model & Subperiod
# =============================================================================

model_summary = (
    subperiod_performance.groupby(["model", "period"], observed=False)[
        metrics_to_aggregate
    ]
    .mean()
    .reset_index()
    .sort_values(["model", "period"])
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("PERFORMANCE BY MODEL ACROSS SUBPERIODS (AVERAGED)")
print("=" * 80)
print(model_summary.to_string(index=False, formatters=formatters))
print("-" * 80)


PERFORMANCE BY MODEL ACROSS SUBPERIODS (AVERAGED)
        model     period   CAGR annualized_volatility Sharpe Sortino maximum_drawdown annualized_turnover
Random Forest  Early OOS 28.99%                30.35%  0.913   1.345          -23.03%             249.76%
Random Forest Middle OOS 44.13%                18.54%  2.295   3.473           -8.88%             248.55%
Random Forest   Late OOS 36.51%                21.32%  1.723   2.619          -11.10%             264.71%
        Ridge  Early OOS 20.77%                28.54%  0.717   1.050          -22.91%             249.59%
        Ridge Middle OOS 47.59%                17.14%  2.732   4.381           -6.64%             273.35%
        Ridge   Late OOS 33.59%                17.47%  1.890   2.915           -9.34%             287.50%
      XGBoost  Early OOS 27.26%                30.34%  0.859   1.262          -23.44%             243.25%
      XGBoost Middle OOS 45.13%                18.38%  2.352   3.573           -8.88%             251

El desglose por subperiodos confirma una marcada heterogeneidad temporal en el perfil de riesgo-retorno, registrándose el régimen de mayor eficiencia en la fase Middle OOS para todas las arquitecturas. En este tramo intermedio, Ridge alcanza el máximo ratio de Sharpe medio del estudio (2.732) y el maximum drawdown más contenido (-6.64%), sustentado en una baja volatilidad anualizada (17.14%).

Sin embargo, en el tramo final (Late OOS), XGBoost exhibe la mayor solidez operativa con un CAGR del 38.48% y un Sharpe de 1.850, superando a Random Forest (Sharpe de 1.723) mediante una menor volatilidad (20.74% frente a 21.32%). El periodo inicial (Early OOS) constituye la fase de mayor estrés de mercado para los tres modelos, caracterizada por volatilidad elevada (28%–30%) y drawdowns superiores al -22%. La incorporación de la deriva de precios (price drift) ajusta los niveles reales de turnover anualizado a un rango sostenido de entre el 243% y el 287%, mostrando un incremento paulatino hacia la fase Late OOS.

In [17]:
# =============================================================================
# Table 2: Performance Grouped by Filtered Portfolio Type & Subperiod
# (Includes Top 10 Long-Only and Long-Short portfolios)
# =============================================================================

# Filter dataset to retain target portfolio architectures
portfolio_pattern = "top_10|long_short|top10|ls"

filtered_subperiod = subperiod_performance[
    subperiod_performance["portfolio"].str.contains(
        portfolio_pattern, case=False, regex=True
    )
].copy()

portfolio_summary = (
    filtered_subperiod.groupby(["portfolio", "period"], observed=False)[
        metrics_to_aggregate
    ]
    .mean()
    .reset_index()
    .sort_values(["portfolio", "period"])
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("PERFORMANCE BY PORTFOLIO ARCHITECTURE ACROSS SUBPERIODS")
print("=" * 80)
print(portfolio_summary.to_string(index=False, formatters=formatters))
print("-" * 80)


PERFORMANCE BY PORTFOLIO ARCHITECTURE ACROSS SUBPERIODS
                          portfolio     period   CAGR annualized_volatility Sharpe Sortino maximum_drawdown annualized_turnover
long_only_top_10_inverse_volatility  Early OOS 30.70%                34.12%  0.883   1.299          -26.75%             298.68%
long_only_top_10_inverse_volatility Middle OOS 53.23%                20.92%  2.559   3.951           -9.29%             299.70%
long_only_top_10_inverse_volatility   Late OOS 45.76%                22.83%  2.019   3.142          -10.20%             296.19%
    long_only_top_10_maximum_sharpe  Early OOS 30.46%                30.34%  1.005   1.479          -24.59%             309.46%
    long_only_top_10_maximum_sharpe Middle OOS 44.55%                19.67%  2.287   3.517           -9.18%             322.85%
    long_only_top_10_maximum_sharpe   Late OOS 32.51%                20.38%  1.604   2.350           -9.57%             316.77%
       long_only_top_10_risk_parity  Early OOS 

El análisis por arquitectura de cartera confirma un sesgo claro entre la generación de retorno absoluto y el control del riesgo. La estrategia signal_weight actúa como el principal motor de retorno, alcanzando en Middle OOS un CAGR del 84.88% y un ratio de Sharpe de 3.537, sustentada por la rotación más contenida del conjunto (233.96%–263.39%). Sin embargo, este sesgo táctico implica asumir la mayor volatilidad (41.91% en Early OOS) y el drawdown máximo más pronunciado (-30.68%).

En el extremo opuesto, la estructura long_short_equal_weight ofrece una clara protección del capital: restringe las caídas máximas al -4.96% en la fase intermedia y comprime la volatilidad anualizada a un rango del 11.63%–15.96%, aunque sacrificando rentabilidad media. Por su parte, los esquemas de optimización pasiva ponderada (inverse_volatility y risk_parity) exhiben una gran consistencia temporal, manteniendo ratios de Sharpe superiores a 2.0 en Late OOS y una rotación estabilizada en torno al 300%–310%.

In [18]:
# =============================================================================
# Table 3: Sharpe Ratio Matrix (Model vs Subperiod Pivot View)
# =============================================================================

sharpe_pivot = subperiod_performance.pivot_table(
    index="model",
    columns="period",
    values="Sharpe",
    aggfunc="mean",
    observed=False,
)[period_order]

# Compute cross-subperiod average and stability metrics (Standard Deviation)
sharpe_pivot["Mean"] = sharpe_pivot.mean(axis=1)
sharpe_pivot["Std Dev"] = sharpe_pivot[period_order].std(axis=1)

print("\n" + "=" * 80)
print("SHARPE RATIO STABILITY MATRIX (MODEL VS SUBPERIOD)")
print("=" * 80)
print(sharpe_pivot.to_string(float_format=lambda x: f"{x:.3f}"))
print("=" * 80)


SHARPE RATIO STABILITY MATRIX (MODEL VS SUBPERIOD)
period         Early OOS  Middle OOS  Late OOS  Mean  Std Dev
model                                                        
Random Forest      0.913       2.295     1.723 1.644    0.694
Ridge              0.717       2.732     1.890 1.779    1.012
XGBoost            0.859       2.352     1.850 1.687    0.760


La matriz de estabilidad del ratio de Sharpe confirma un comportamiento temporal coherente entre las tres arquitecturas, respaldando la validez del proceso de aprendizaje fuera de muestra. La dinámica por subperiodos exhibe un patrón común: una fase inicial exigente (Early OOS, con promedios entre 0.717 y 0.913), un repunte de eficiencia ajustada por riesgo en el tramo intermedio (Middle OOS, donde todas las arquitecturas superan la cota de 2.290), y una convergencia hacia niveles sostenibles en el periodo final (Late OOS).

Ridge alcanza el promedio global de Sharpe más elevado (1.779), aunque presenta la mayor dispersión intertemporal ($\text{Std Dev} = 1.012$) debido a su marcada sensibilidad al régimen de mercado de Middle OOS (2.732). Por el contrario, Random Forest demuestra el perfil más estable y predecible ($\text{Std Dev} = 0.694$), manteniendo un rendimiento homogéneo a través de las distintas fases temporales. XGBoost ofrece un punto de equilibrio óptimo, combinando un Sharpe medio elevado (1.687) con una degradación contenida en el tramo final (Late OOS de 1.850 frente al 1.723 de Random Forest).

El análisis de estabilidad temporal demuestra que la generación de alfa basada en Machine Learning es sólida y consistente a lo largo del tiempo. La combinación de XGBoost con un esquema de ponderación por intensidad de señal (signal weight) —para maximizar el retorno absoluto— o con arquitecturas de paridad de riesgo / long-short —para inmunizar la cartera ante episodios de estrés— se consolida como la solución óptima, equilibrando de forma eficiente la rentabilidad neta, la sostenibilidad intertemporal y la reducción de los costes de ejecución tras considerar la deriva de precios.

## 7. Benchmark Comparison, Alpha Attribution & Rolling Performance

Una vez evaluada la estabilidad temporal del sistema a través de las distintas fases *out-of-sample* (OOS) y analizada la sensibilidad del rendimiento neto ante cambios en la frecuencia de rebalanceo, este capítulo aborda la evaluación económica definitiva del proyecto.

Evaluar una estrategia cuantitativa de forma aislada —por ejemplo, observando únicamente una rentabilidad anualizada absoluta— resulta metodológicamente insuficiente en la gestión de activos moderna. Un rendimiento elevado no implica de forma automática la presencia de habilidad predictiva o superioridad técnica; dicho retorno puede ser el simple reflejo de una exposición pasiva a factores de riesgo sistemático (*Beta* de mercado) o de un comportamiento atípico concentrado en un periodo temporal reducido.

Para validar si el sistema cuantitativo diseñado genera **alfa genuino** (entendido como exceso de rentabilidad ajustado por riesgo y despojado de primas de riesgo tradicionales), este bloque articula un marco de evaluación en cinco fases complementarias:

1. **Formalización de la muestra de carteras candidatas** para enfocar el análisis de atribución en configuraciones representativas del espacio de soluciones.

2. **Descomposición causal del rendimiento** mediante *benchmarking* por componentes, aislando el impacto del algoritmo de *Machine Learning*, la influencia de factores clásicos (*Momentum*) y el aporte del motor de optimización de carteras.

3. **Validación de significación estadística mediante simulación estocástica de Monte Carlo**, contrastando el desempeño del sistema frente a carteras pseudo-aleatorias equivalentes.

4. **Cuantificación de métricas de gestión activa** (*Tracking Error* e *Information Ratio*), evaluando la eficiencia en la conversión de riesgo activo en exceso de retorno.

5. **Diagnóstico de consistencia temporal continua** a través de métricas móviles (*Rolling Performance*), perfiles de *drawdown* históricos y desglose por años naturales.

### 7.1 Candidate Strategy Selection

El proceso de optimización y simulación desarrollado en los capítulos anteriores ha generado un universo amplio de combinaciones resultantes de cruzar arquitecturas de aprendizaje automático (*Ridge*, *Random Forest*, *XGBoost*), esquemas de ponderación (*Signal Weighting*, *Equal Weight*, *Inverse Volatility*, *Risk Parity*, *Maximum Sharpe*, *Long-Short*) y horizontes de selección de activos (*Top 10%*, *Top 20%*, *Top 30%*).

Procesar la totalidad de estas series en las pruebas complejas de atribución de alfa y simulación de Monte Carlo no solo resultaría ineficiente desde el punto de vista computacional, sino que diluiría la claridad ejecutiva del análisis. Por tanto, se establece un filtro transparente y fundamentado para seleccionar **cuatro carteras candidatas representativas** que sintetizan la frontera eficiente del estudio:

* **Estrategia de Máxima Eficiencia (*Highest Sharpe Candidate*):** Representa el punto de óptima conversión de riesgo en retorno en la muestra completa. Esta cartera permite evaluar la capacidad del sistema para maximizar el exceso de rentabilidad por unidad de volatilidad total, minimizando el impacto del *churning* y las fricciones de ejecución.

* **Estrategia de Máximo Retorno Absoluto (*Highest CAGR Candidate*):** Selecciona la configuración que ha liderado la generación bruta y neta de capital a lo largo del periodo *out-of-sample*. Esta alternativa explota la máxima intensidad predictiva del modelo, asumiendo una mayor concentración y volatilidad a cambio de capturar la cola superior de la distribución de retornos (*tail alpha*).

* **Estrategia Defensiva y Control de Riesgo (*Defensive / High Calmar Candidate*):** Escoge la arquitectura enfocada en la preservación del patrimonio (caracterizada por un *Maximum Drawdown* acotado y una rápida velocidad de recuperación). Esta selección evalúa la capacidad de las estrategias neutrales o de baja varianza para inmunizar el capital durante fases de tensión en los mercados financieros.

* **Estrategia Base ML (*Baseline ML Candidate*):** Definida como la combinación de las predicciones del modelo con una asignación pasiva de igual ponderación (*Top 10% Equal Weight*). Esta cartera actúa como el eslabón fundamental de control, permitiendo separar la habilidad puramente predictiva del algoritmo de cualquier sesgo introducido por las técnicas avanzadas de construcción de carteras.

In [19]:
# =============================================================================
# 7.1 Candidate Strategy Selection
# =============================================================================

# -------------------------------------------------------------------------
# Configuration & Constants
# -------------------------------------------------------------------------
FREQUENCY_SENSITIVITY_PATH = (
    "../data/portfolio_results/"
    "rebalancing_frequency_sensitivity_metrics.parquet"
)

TOP_N_CANDIDATES = 5

DISPLAY_COLUMNS = [
    "frequency_days",
    "model",
    "portfolio",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",  # Critical metric for execution viability
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

FORMATTERS = {
    "CAGR": lambda x: f"{x:.2%}",
    "annualized_volatility": lambda x: f"{x:.2%}",
    "annualized_turnover": lambda x: f"{x:.2%}",
    "Sharpe": lambda x: f"{x:.3f}",
    "Sortino": lambda x: f"{x:.3f}",
    "Calmar": lambda x: f"{x:.3f}",
    "maximum_drawdown": lambda x: f"{x:.2%}",
}

# -------------------------------------------------------------------------
# Load & Validate Data
# -------------------------------------------------------------------------
df = pd.read_parquet(FREQUENCY_SENSITIVITY_PATH)
df["frequency_days"] = df["frequency_days"].astype(int)

required_cols = DISPLAY_COLUMNS + ["frequency_label"]
missing_cols = [col for col in required_cols if col not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"
assert df[["CAGR", "Sharpe", "Calmar", "annualized_turnover"]].notna().all().all()


# -------------------------------------------------------------------------
# Helper Function for Top N Extraction & Printing
# -------------------------------------------------------------------------

from src.portfolio.utils import display_top_candidates

# -------------------------------------------------------------------------
# 1. Evaluate Shortlists for Key Criteria
# -------------------------------------------------------------------------
top_sharpe = display_top_candidates(
    df, "Sharpe", "Highest Sharpe Candidates (Max Efficiency)"
)
top_cagr = display_top_candidates(
    df, "CAGR", "Highest CAGR Candidates (Max Return)"
)
top_calmar = display_top_candidates(
    df, "Calmar", "Highest Calmar Candidates (Defensive / Risk Control)"
)


TOP 5 — HIGHEST SHARPE CANDIDATES (MAX EFFICIENCY)
 frequency_days         model                      portfolio   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
             21 Random Forest long_only_top_10_signal_weight 89.75%                36.28%             219.69%  2.474   3.663  2.764          -32.47%
             21       XGBoost long_only_top_10_signal_weight 81.63%                35.35%             234.84%  2.309   3.412  2.522          -32.37%
             21       XGBoost        long_short_equal_weight 32.45%                14.54%             199.28%  2.232   3.238  2.525          -12.85%
             21 Random Forest         long_only_equal_weight 72.42%                32.85%             233.38%  2.205   3.279  2.378          -30.45%
             21       XGBoost long_only_top_20_signal_weight 68.14%                31.00%             227.14%  2.198   3.293  2.343          -29.09%

TOP 5 — HIGHEST CAGR CANDIDATES (MAX RETURN)
 frequen

In [20]:
# =============================================================================
# 1. Define Selected Candidate Strategies
# =============================================================================

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)

df_all_metrics = pd.read_parquet(METRICS_PATH)

CANDIDATE_KEYS = [
    {
        "role": "Highest Sharpe",
        "model": "Random Forest",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest CAGR",
        "model": "XGBoost",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest Calmar",
        "model": "XGBoost",
        "portfolio": "long_short_equal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Baseline ML",
        "model": "Random Forest",
        "portfolio": "long_only_equal_weight",
        "frequency_days": 21,
    },
]

# Merge metrics for audit output
candidates_keys_df = pd.DataFrame(CANDIDATE_KEYS)

selected_candidates_table = candidates_keys_df.merge(
    df_all_metrics,
    on=["model", "portfolio", "frequency_days"],
    how="inner",
)

# =============================================================================
# 2. Audit Output — 4 Selected Candidates
# =============================================================================

print("=" * 110)
print("SELECTED CANDIDATE STRATEGIES")
print("=" * 110)

print(
    selected_candidates_table[
        [
            "role",
            "model",
            "portfolio",
            "frequency_days",
            "CAGR",
            "annualized_volatility",
            "annualized_turnover",  # Added metric
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
        ]
    ].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "annualized_turnover": lambda x: f"{x:.2%}",  # Added formatter
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

# =============================================================================
# 3. Extract Target & Executed Weights for Selected Candidates (MultiIndex)
# =============================================================================

candidate_target_dict = {}
candidate_executed_dict = {}

all_weight_dates = (
    weights_raw["date"].drop_duplicates().sort_values().reset_index(drop=True)
)

# daily_returns_matrix must be a DataFrame (index=date, columns=ticker) with daily asset returns
for config in CANDIDATE_KEYS:
    role = config["role"]
    model = config["model"]
    portfolio = config["portfolio"]
    frequency = config["frequency_days"]

    key = (role, model, portfolio, frequency)

    # 1. Rebalancing calendar
    rebalancing_dates = all_weight_dates.iloc[::frequency]

    # 2. Filter raw weights for target matrix
    mask = (
        (weights_raw["model"] == model)
        & (weights_raw["portfolio"] == portfolio)
        & (weights_raw["date"].isin(rebalancing_dates))
    )
    group = weights_raw[mask].copy()

    # Target Weights Matrix (Rebalance dates only)
    target_weight_matrix = (
        group.pivot(index="date", columns="ticker", values="weight")
        .fillna(0.0)
        .reindex(index=rebalancing_dates, columns=simple_returns.columns)
        .fillna(0.0)
    )

    # 3. Simulate Daily Executed Weights with Price Drift (Buy & Hold)
    executed_weights_list = []

    # Get execution dates (T+1 from target rebalance date)
    for i in range(len(rebalancing_dates)):
        reb_date = rebalancing_dates.iloc[i]
        
        # Next rebalance date defines the end of current holding period
        next_reb_date = (
            rebalancing_dates.iloc[i + 1]
            if i + 1 < len(rebalancing_dates)
            else all_trading_dates[-1]
        )

        # Execution starts at T+1 relative to target decision date
        reb_idx = all_trading_dates.get_loc(reb_date)
        if reb_idx + 1 >= len(all_trading_dates):
            break
            
        exec_start_date = all_trading_dates[reb_idx + 1]

        # Extract returns for holding window [T+1, next_reb_date]
        window_returns = simple_returns.loc[exec_start_date:next_reb_date]
        
        # Initial target weight at T+1
        w_current = target_weight_matrix.loc[reb_date].values

        # Track weights over the drift period
        for date, ret in window_returns.iterrows():
            executed_weights_list.append(
                pd.Series(w_current, index=simple_returns.columns, name=date)
            )
            # Update weights based on asset returns (Drift)
            w_gross = w_current * (1.0 + ret.values)
            total_gross = w_gross.sum()
            
            # Avoid division by zero if cash/empty
            w_current = w_gross / total_gross if total_gross != 0 else w_gross

    executed_matrix = pd.DataFrame(executed_weights_list)

    # Save outputs
    candidate_target_dict[key] = target_weight_matrix
    candidate_executed_dict[key] = executed_matrix

# =============================================================================
# Consolidate into MultiIndex DataFrames
# =============================================================================

index_names = ["role", "model", "portfolio", "frequency_days", "date"]

df_target_weights = pd.concat(candidate_target_dict, names=index_names).fillna(0.0)
df_executed_weights = pd.concat(candidate_executed_dict, names=index_names).fillna(0.0)

print("\n" + "=" * 110)
print("✓ Daily executed weights correctly calculated with asset drift (T+1).")
print("=" * 110)

SELECTED CANDIDATE STRATEGIES
          role         model                      portfolio  frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
Highest Sharpe Random Forest long_only_top_10_signal_weight              21 89.75%                36.28%             219.69%  2.474   3.663  2.764          -32.47%
  Highest CAGR       XGBoost long_only_top_10_signal_weight              21 81.63%                35.35%             234.84%  2.309   3.412  2.522          -32.37%
Highest Calmar       XGBoost        long_short_equal_weight              21 32.45%                14.54%             199.28%  2.232   3.238  2.525          -12.85%
   Baseline ML Random Forest         long_only_equal_weight              21 72.42%                32.85%             233.38%  2.205   3.279  2.378          -30.45%

✓ Daily executed weights correctly calculated with asset drift (T+1).


Para evitar redundancias computacionales y mantener una narrativa coherente en las pruebas de estrés, atribución de alfa y simulación de Monte Carlo, se filtran las estrategias evaluadas mediante criterios de eficiencia, captura de retorno, preservación de capital y aislamiento de la señal ML. La selección de candidatos evita repetir exactamente la misma serie temporal bajo distintas etiquetas —situación que se daría aplicando un argmax matemático ciego— y prioriza la viabilidad operativa dada por el nivel de turnover anualizado:

- **Estrategia de Máxima Eficiencia (Highest Sharpe Candidate)**: Representada por la arquitectura long_only_top_10_signal_weight bajo el algoritmo Random Forest a 21 días de rebalanceo. Esta alternativa se sitúa en el Top 1 absoluto de conversión de riesgo en retorno con un Ratio de Sharpe de 2.474, un CAGR del 89.75%, un Ratio de Calmar de 2.764 y un Maximum Drawdown del -32.47%. Su inclusión se justifica al marcar el techo analítico de eficiencia total del sistema, demostrando la capacidad de los árboles de decisión combinados con una asignación ponderada por la intensidad de la señal para capturar exceso de rentabilidad ajustada por riesgo.

- **Estrategia de Máximo Retorno Absoluto (Highest CAGR Candidate)**: Asignada a la configuración long_only_top_10_signal_weight con el modelo XGBoost a 21 días de rebalanceo. Al ser el primer puesto de CAGR la serie elegida anteriormente, se selecciona el segundo mejor valor del estudio para evitar la duplicación de datos. Esta cartera alcanza un CAGR del 81.63%, acompañado de un Ratio de Sharpe de 2.309, un Ratio de Calmar de 2.522 y una caída máxima del -32.37%. Su selección permite evaluar la intensidad predictiva del gradient boosting en un esquema de alta concentración, asumiendo un perfil de varianza elevado (volatilidad del 35.35%) ideal para estresar el Tracking Error y el Information Ratio.

- **Estrategia Defensiva y de Control de Riesgo (Defensive / High Calmar Candidate)**: Materializada en la cartera long_short_equal_weight mediante el algoritmo XGBoost a 21 días. Frente al argmax literal de Calmar —que devolvería una cartera concentrada con caídas superiores al -32%—, esta elección responde a la verdadera naturaleza defensiva descrita en la metodología: comprime el Maximum Drawdown a un -12.85% y la volatilidad a un nivel del 14.54%, registrando un CAGR del 32.45% y un Sharpe de 2.232. La elección de XGBoost sobre otras variantes defensivas se justifica por su menor rotación anualizada (199.28%), ofreciendo una alternativa con alta protección frente a fases de tensión en los mercados y de menor fricción operativa.

- **Estrategia Base ML (Baseline ML Candidate)**: Formalizada mediante la estructura long_only_equal_weight bajo el modelo Random Forest a 21 días. Con un CAGR del 72.42%, un Ratio de Sharpe de 2.205, una volatilidad del 32.85% y un Maximum Drawdown del -30.45%, esta cartera combina el universo seleccionado por la señal ML con una ponderación pasiva de igual peso. Mantener el mismo modelo (Random Forest) que el candidato de Máxima Eficiencia permite aislar el efecto de la ponderación táctica (Signal Weighting vs. Equal Weight), sirviendo como control en la posterior atribución de alfa.

Para optimizar la carga computacional en las fases de atribución, las matrices de exposición de las cuatro candidatas se han persistido en disco en formato Parquet a través de dos estructuras complementarias.

Por una parte, los pesos objetivo (target_weights) registran la asignación táctica teórica dictada por el modelo exclusivamente en las fechas de rebalanceo, lo que permite auditar la decisión pura del algoritmo y la rotación de activos. Por otra parte, los pesos ejecutados (executed_weights) proyectan la exposición diaria real de la cartera al aplicar un desfase operativo de un día ($T+1$) tras la señal para eliminar el sesgo de premonición (look-ahead bias), manteniendo las posiciones mediante un esquema buy-and-hold hasta la siguiente ventana. Esta última matriz constituye el insumo principal para el cálculo continuo de retornos, alfas y Tracking Error.

In [21]:
df_target_weights.to_parquet("../data/portfolio_results/candidate_target_weights.parquet")
df_executed_weights.to_parquet("../data/portfolio_results/candidate_executed_weights.parquet")

### 7.2 Absolute & Factor-Adjusted Benchmarking

Para determinar con rigurosidad científica la procedencia del rendimiento, no basta con demostrar que el sistema completo obtiene plusvalías. Es indispensable responder a tres preguntas metodológicas esenciales: *¿La estrategia supera la exposición pasiva al mercado?*, *¿El rendimiento se debe al algoritmo de Machine Learning o al factor de riesgo subyacente?* y *¿Cuánto valor añade realmente la etapa de optimización de pesos?*

Para aislar el efecto de cada decisión de diseño dentro de la cadena operativa:

$$\text{Predicción ML} \longrightarrow \text{Selección de Señal} \longrightarrow \text{Ponderación de Cartera} \longrightarrow \text{Estrategia Final}$$

se estructuran cuatro *benchmarks* de contraste.


#### 7.2.1 Market Baseline Comparison (Benchmark A)

La primera referencia de evaluación es la estrategia pasiva de comprar y mantener el índice de mercado:

$$\text{Benchmark A} = \text{S\&P 500 Buy \& Hold}$$

Esta comparativa responde a una de las preguntas fundamentales del estudio: ¿genera la gestión activa basada en señales de Machine Learning suficiente valor añadido frente a la mera exposición pasiva al mercado como para justificar la complejidad adicional?

Las carteras candidatas se evalúan frente al índice de referencia utilizando métricas normalizadas de crecimiento patrimonial y comportamiento ajustado por riesgo: Compound Annual Growth Rate (CAGR), Volatilidad Anualizada ($\sigma_{\text{ann}}$), Sharpe Ratio, Sortino Ratio y Maximum Drawdown (MDD). Adicionalmente, se incorpora la Rentabilidad Acumulada (Cumulative Return) para comparar directamente la evolución patrimonial durante todo el periodo out-of-sample.

Para garantizar una comparación económicamente homogénea, las métricas de las estrategias ML se calculan sobre rentabilidades netas de costes de transacción, mientras que el benchmark representa una estrategia buy & hold con una fricción operativa mínima.

El análisis permitirá determinar no solo qué estrategias superan al mercado en términos de rentabilidad absoluta, sino también si dicha superioridad se obtiene mediante una asunción de riesgo significativamente mayor o si las estrategias ML consiguen mejorar simultáneamente la eficiencia riesgo-retorno y el control de pérdidas.


In [22]:
# =============================================================================
# S&P 500 Benchmark — Data Extraction
# =============================================================================

BENCHMARK_TICKER = "^GSPC"

OOS_START_DATE = "2025-01-15"
OOS_END_DATE = "2026-08-11"

benchmark_data = yf.download(
    BENCHMARK_TICKER,
    start=OOS_START_DATE,
    end=OOS_END_DATE,
    auto_adjust=False,
    progress=False,
)

# -----------------------------------------------------------------------------
# Extract Adjusted Close
# -----------------------------------------------------------------------------

benchmark_prices = benchmark_data["Adj Close"]

# yfinance may return a DataFrame even for a single ticker
if isinstance(benchmark_prices, pd.DataFrame):
    benchmark_prices = benchmark_prices.iloc[:, 0]

benchmark_prices = benchmark_prices.copy()

# -----------------------------------------------------------------------------
# Index Formatting
# -----------------------------------------------------------------------------

benchmark_prices.index = pd.to_datetime(
    benchmark_prices.index
)

if benchmark_prices.index.tz is not None:
    benchmark_prices.index = (
        benchmark_prices.index.tz_localize(None)
    )

benchmark_prices.name = "sp500_adj_close"

# =============================================================================
# Audit
# =============================================================================

print("=" * 80)
print("S&P 500 BENCHMARK — DATA EXTRACTION")
print("=" * 80)

print(
    f"✓ Ticker = {BENCHMARK_TICKER}"
)

print(
    f"✓ Date range = "
    f"{benchmark_prices.index.min().date()} "
    f"→ "
    f"{benchmark_prices.index.max().date()}"
)

print(
    f"✓ Observations = "
    f"{len(benchmark_prices):,}"
)

print(
    f"✓ Missing prices = "
    f"{benchmark_prices.isna().sum():,}"
)

print(
    f"✓ Data type = "
    f"{type(benchmark_prices).__name__}"
)

# =============================================================================
# S&P 500 — Daily Returns
# =============================================================================

benchmark_returns = (
    benchmark_prices
    .pct_change()
    .dropna()
    .rename("benchmark_return")
)

print()
print("=" * 80)
print("S&P 500 BENCHMARK — RETURN SERIES")
print("=" * 80)

print(
    f"✓ Return observations = "
    f"{len(benchmark_returns):,}"
)

print(
    f"✓ First return date = "
    f"{benchmark_returns.index.min().date()}"
)

print(
    f"✓ Last return date = "
    f"{benchmark_returns.index.max().date()}"
)

print(
    f"✓ Missing returns = "
    f"{benchmark_returns.isna().sum():,}"
)

S&P 500 BENCHMARK — DATA EXTRACTION
✓ Ticker = ^GSPC
✓ Date range = 2025-01-15 → 2026-08-10
✓ Observations = 393
✓ Missing prices = 0
✓ Data type = Series

S&P 500 BENCHMARK — RETURN SERIES
✓ Return observations = 392
✓ First return date = 2025-01-16
✓ Last return date = 2026-08-10
✓ Missing returns = 0


In [23]:
from src.portfolio.metrics import calculate_benchmark_metrics


# =============================================================================
# Global Constants & Configuration
# =============================================================================

TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0

CANDIDATES = [
    {
        "role": "Highest Sharpe",
        "model": "Random Forest",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest CAGR",
        "model": "XGBoost",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest Calmar",
        "model": "XGBoost",
        "portfolio": "long_short_equal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Baseline ML",
        "model": "Random Forest",
        "portfolio": "long_only_equal_weight",
        "frequency_days": 21,
    },
]

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)

# =============================================================================
# Main Execution Pipeline
# =============================================================================

# 1. Load precomputed frequency sensitivity metrics
df_all_metrics = pd.read_parquet(METRICS_PATH)

# 2. Extract metrics for selected candidate strategies
candidate_records = []

for candidate in CANDIDATES:
    mask = (
        (df_all_metrics["model"] == candidate["model"])
        & (df_all_metrics["portfolio"] == candidate["portfolio"])
        & (df_all_metrics["frequency_days"] == candidate["frequency_days"])
    )

    row = df_all_metrics.loc[mask].copy()

    if not row.empty:
        row["role"] = candidate["role"]
        candidate_records.append(row)

df_candidates_metrics = pd.concat(candidate_records, ignore_index=True)

# 3. Compute benchmark metrics (expects `benchmark_returns` in current scope)
benchmark_metrics = calculate_benchmark_metrics(benchmark_returns)
benchmark_row = pd.DataFrame([benchmark_metrics])

# 4. Consolidate and apply explicit ordering
market_baseline_comparison = pd.concat(
    [benchmark_row, df_candidates_metrics], ignore_index=True
)

role_order = [
    "Benchmark",
    "Highest Sharpe",
    "Highest CAGR",
    "Highest Calmar",
    "Baseline ML",
]

market_baseline_comparison["role"] = pd.Categorical(
    market_baseline_comparison["role"], categories=role_order, ordered=True
)

market_baseline_comparison = market_baseline_comparison.sort_values(
    "role"
).reset_index(drop=True)

# 5. Formatted audit output
cols_to_show = [
    "role",
    "model",
    "portfolio",
    "frequency_days",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

market_baseline_comparison["frequency_days"] = (
    market_baseline_comparison["frequency_days"]
    .map(lambda x: f"{int(x)}" if pd.notnull(x) else "N/A")
)

print("=" * 110)
print("MARKET BASELINE COMPARISON")
print("=" * 110)

print(
    market_baseline_comparison[cols_to_show].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "annualized_turnover": lambda x: (
                f"{x:.2%}" if pd.notnull(x) else "N/A"
            ),
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

MARKET BASELINE COMPARISON
          role         model                      portfolio frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
     Benchmark       S&P 500                   buy_and_hold            N/A 18.55%                17.10%               0.00%  1.085   1.615  0.981          -18.90%
Highest Sharpe Random Forest long_only_top_10_signal_weight             21 89.75%                36.28%             219.69%  2.474   3.663  2.764          -32.47%
  Highest CAGR       XGBoost long_only_top_10_signal_weight             21 81.63%                35.35%             234.84%  2.309   3.412  2.522          -32.37%
Highest Calmar       XGBoost        long_short_equal_weight             21 32.45%                14.54%             199.28%  2.232   3.238  2.525          -12.85%
   Baseline ML Random Forest         long_only_equal_weight             21 72.42%                32.85%             233.38%  2.205   3.279  2.378          -30

Los resultados comparativos confirman que las tres estrategias seleccionadas —Highest Sharpe, Highest CAGR y Highest Calmar— superan con solvencia al S&P 500 (18,55% CAGR; Sharpe 1,085) durante el periodo 2025–2026. Asimismo, logran batir al Baseline ML (72,42% CAGR; Sharpe 2,205), manteniendo Ratios de Sharpe competitivos y superiores al modelo base en las variantes top 10 signal weight (2,474 y 2,309).

A pesar del excelente desempeño absoluto, destacan dos factores de riesgo a vigilar. Por un lado, la alta volatilidad de las estrategias concentradas (35%–36%) genera caídas severas, alcanzando un Maximum Drawdown del -32,47% en la variante de mayor Sharpe (frente al -18,90% del índice), siendo Highest Calmar la única capaz de contener la pérdida máxima (-12,85%) con una volatilidad contenida (14,54%). Por otro lado, la rotación anualizada es elevada en todas las carteras activas —oscilando entre el 199,28% y el 234,84% a 21 días—, lo que exige vigilar estrechamente la erosión causada por los costes operativos.


#### 7.2.2 Component-Level Benchmarking & Alpha Attribution

Si una estrategia basada en *Machine Learning* utiliza como variables de entrada indicadores técnicos de tendencia o retornos pasados, existe el riesgo de que el modelo actúe simplemente como un *proxy* complejo de un factor sistemático tradicional, como el **Momentum**. Si comprar directamente el decil con mayor *Momentum* ofrece un resultado equivalente al del modelo ML, el mérito de la rentabilidad debe atribuirse a la prima de riesgo del factor y no a la capacidad analítica del algoritmo.

Para desacoplar el origen del alfa y medir el valor añadido de cada componente del *pipeline*, se definen tres *benchmarks* analíticos:


##### Benchmark B — Momentum Factor + Equal Weight (Evaluación del Algoritmo ML)

Se construye una cartera que selecciona el $10\%$ de los activos con mayor rentabilidad acumulada en la ventana de predicción ($21$ días) y los pondera de forma equitativa:

$$\text{Benchmark B} = \text{Momentum (Top 10\%)} \longrightarrow \text{Equal Weight}$$

Al contrastar la estrategia candidata base $\text{ML (Top 10\%)} \rightarrow \text{Equal Weight}$ frente al $\text{Benchmark B}$, se elimina el efecto de la asignación de pesos y se aísla exclusivamente la capacidad discriminatoria del modelo ML frente a la regla mecánica o simple de *momentum* cross-sectional.

Cabe precisar la decisión de emplear un factor de **Momentum de horizonte corto (21-day Momentum)** en lugar del estándar académico de medio plazo (**12–1 Month Momentum**). Aunque el histórico previo permite construir la señal clásica, evaluar un factor de 12 meses sobre una ventana *out-of-sample* (OOS) de solo ~18 meses (2025–2026) expondría la comparativa a un alto sesgo por el régimen de mercado concreto de ese periodo.

Además, prima la coherencia metodológica: como los modelos de ML predicen la rentabilidad esperada a un horizonte de 21 días, enfrentarlos a inercias de 12 meses desvirtuaría el test de atribución. No se estaría evaluando si el ML selecciona mejor que una regla simple, sino comparando dos horizontes de inversión distintos.

Para garantizar la máxima neutralidad en la comparativa, la cartera del Benchmark B se construye bajo el mismo marco de restricciones operativas que las carteras de Machine Learning. Esto implica aplicar los mismos límites de ponderación individual (pesos máximo y mínimo por activo) y someter el rebalanceo a la misma restricción de rotación (turnover limit). De esta forma, se elimina cualquier distorsión derivada de la ejecución o de la gestión del riesgo, garantizando que las diferencias observadas en el desempeño provengan exclusivamente de la capacidad predictiva del algoritmo frente al factor mecánico.


In [24]:
# =============================================================================
# Configuration & Constants
# =============================================================================

MOMENTUM_LOOKBACK = 21
SELECTION_PERCENT = 0.10
MAX_TURNOVER_DESIGN = 0.25
MIN_WEIGHT = 0.005
MAX_WEIGHT = 0.05

TRADING_DAYS_PER_YEAR = 252
REBALANCING_DAYS = 21
REBALANCINGS_PER_YEAR = TRADING_DAYS_PER_YEAR / REBALANCING_DAYS


# =============================================================================
# Load Data & Target Schedule
# =============================================================================

prices = pd.read_parquet("../data/raw/sp500_prices_extended.parquet")
prices.index = pd.to_datetime(prices.index)
adj_close = prices["Adj Close"].copy()

df_target_weights = pd.read_parquet(
    "../data/portfolio_results/candidate_target_weights.parquet"
)

df_temp = df_target_weights.reset_index()

baseline_mask = (
    (df_temp["role"] == "Baseline ML")
    & (df_temp["model"] == "Random Forest")
    & (df_temp["portfolio"] == "long_only_equal_weight")
    & (df_temp["frequency_days"] == 21)
)

rebalancing_dates = (
    df_temp.loc[baseline_mask, "date"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

# =============================================================================
# Load Simple Returns required for Drift Calculation
# =============================================================================

# Calculated from daily price matrix
simple_returns = adj_close.pct_change().fillna(0.0)

# =============================================================================
# Step 1: Compute Target Weights (21-Day Momentum + Equal Weight Top 10%)
# =============================================================================

# 1. Compute 21-day rolling cumulative returns (Momentum signal)
momentum_signal = adj_close.pct_change(MOMENTUM_LOOKBACK)

target_weights_list = []

for reb_date in rebalancing_dates:
    # Get available cross-sectional momentum signals up to rebalancing date T
    if reb_date not in momentum_signal.index:
        continue

    signals_at_date = momentum_signal.loc[reb_date].dropna()

    # Determine top 10% threshold count
    n_assets = len(signals_at_date)
    top_n = max(1, int(np.floor(n_assets * SELECTION_PERCENT)))

    # Select top N tickers with highest 21-day cumulative return
    top_tickers = signals_at_date.nlargest(top_n).index

    # Assign raw equal weights (1 / N)
    raw_weights = pd.Series(0.0, index=adj_close.columns, name=reb_date)
    if top_n > 0:
        raw_weights[top_tickers] = 1.0 / top_n

    target_weights_list.append(raw_weights)

# Consolidate raw target weights matrix (Rebalance dates x Tickers)
df_bmk_target_raw = pd.DataFrame(target_weights_list)


In [25]:

# =============================================================================
# Step 2: Apply Constraints & Audit Benchmark B
# =============================================================================

from src.portfolio.benchmarks import apply_benchmark_constraints_with_drift

df_bmk_target_constrained, df_bmk_executed_daily = (
    apply_benchmark_constraints_with_drift(
        target_weights_wide=df_bmk_target_raw,
        simple_returns=simple_returns,
        max_turnover_design=MAX_TURNOVER_DESIGN,
        min_weight=MIN_WEIGHT,
        max_weight=MAX_WEIGHT,
    )
)

# 2. Compute Realized Turnover at each rebalance date considering Asset Drift
rebalance_dates = df_bmk_target_constrained.index
turnover_series = []

for i in range(1, len(rebalance_dates)):
    prev_reb = rebalance_dates[i - 1]
    curr_reb = rebalance_dates[i]

    # Portfolio before rebalance (previous position drifted by market returns)
    prev_weights = df_bmk_target_constrained.loc[prev_reb]
    window_rets = simple_returns.loc[
        (simple_returns.index > prev_reb) & (simple_returns.index <= curr_reb)
    ]
    compounded = (1.0 + window_rets).prod() - 1.0
    compounded = compounded.reindex(prev_weights.index).fillna(0.0)

    drifted = prev_weights * (1.0 + compounded)
    drifted_sum = drifted.sum()
    drifted_norm = (
        drifted / drifted_sum if drifted_sum > 0 else prev_weights
    )

    # New target portfolio after rebalance constraints
    curr_weights = df_bmk_target_constrained.loc[curr_reb]

    # Realized turnover calculation
    realized_turnover = 0.5 * (curr_weights - drifted_norm).abs().sum()
    turnover_series.append(realized_turnover)

turnover_series = pd.Series(turnover_series, index=rebalance_dates[1:])

# 3. Extract position weight statistics (excluding 0.0 unallocated weights)
active_target_weights = df_bmk_target_constrained.replace(0.0, np.nan)
active_executed_weights = df_bmk_executed_daily.replace(0.0, np.nan)

# =============================================================================
# Print Audit Report
# =============================================================================
print("=" * 80)
print("BENCHMARK B — CONSTRAINTS & DRIFT AUDIT REPORT")
print("=" * 80)

print(f"\n1. TURNOVER METRICS (accounting for Asset Price Drift):")
print(f"   • Rebalances evaluated: {len(turnover_series)}")
print(f"   • Max Turnover per event: {turnover_series.max():.2%}")
print(f"   • Mean Turnover per event: {turnover_series.mean():.2%}")
print(
    f"   • Annualized Turnover:     {turnover_series.mean() * REBALANCINGS_PER_YEAR:.2%}"
)

print(f"\n2. TARGET WEIGHTS BOX CONSTRAINTS (at Rebalance dates T):")
print(f"   • Max Position Weight:     {active_target_weights.max().max():.2%}")
print(f"   • Min Active Weight:       {active_target_weights.min().min():.2%}")

print(f"\n3. EXECUTED WEIGHTS BOX CONSTRAINTS (Daily T+1 with Drift):")
print(f"   • Max Position Weight:     {active_executed_weights.max().max():.2%}")
print(f"   • Min Active Weight:       {active_executed_weights.min().min():.2%}")
print("=" * 80)

BENCHMARK B — CONSTRAINTS & DRIFT AUDIT REPORT

1. TURNOVER METRICS (accounting for Asset Price Drift):
   • Rebalances evaluated: 17
   • Max Turnover per event: 39.18%
   • Mean Turnover per event: 32.58%
   • Annualized Turnover:     390.92%

2. TARGET WEIGHTS BOX CONSTRAINTS (at Rebalance dates T):
   • Max Position Weight:     5.00%
   • Min Active Weight:       0.53%

3. EXECUTED WEIGHTS BOX CONSTRAINTS (Daily T+1 with Drift):
   • Max Position Weight:     7.60%
   • Min Active Weight:       0.30%


La auditoría operativa del Benchmark B valida el cumplimiento estricto de las restricciones tácticas a nivel de activo en las fechas de rebalanceo ($T$), alcanzando una ponderación máxima del 5,00% y respetando el umbral mínimo efectivo del 0,53%. En el seguimiento diario con deriva pasiva ($T+1$), la exposición máxima individual se expande temporalmente hasta el 7,60% como reflejo natural del *asset price drift* entre ventanas de negociación. 

En materia de rotación, la estrategia registra un *turnover* medio por evento del 32,58% (390,92% anualizado). La desviación observada frente al parámetro nominal de diseño (25,00%) no responde a un fallo de acotación, sino a la renormalización secundaria del capital derivada de la aplicación posterior de los límites individuales (*box constraints*). Este resultado evidencia la inestabilidad inherente al factor *momentum* mecánico a 21 días, el cual requiere una rotación notablemente superior a la de los modelos ML (~199%–235% anualizado) para operar bajo el mismo marco de restricciones de riesgo.

In [26]:
# =============================================================================
# Transaction Costs Configuration
# =============================================================================

TRANSACTION_COST_BPS_BASE = 15.0
BPS_TO_DECIMAL = 10_000.0

TRADING_DAYS_PER_YEAR = 252
REBALANCING_DAYS = 21
REBALANCINGS_PER_YEAR = TRADING_DAYS_PER_YEAR / REBALANCING_DAYS

# =============================================================================
# Compute Rebalance-Level Turnover (Accounting for Asset Drift)
# =============================================================================

rebalance_dates = df_bmk_target_constrained.index.sort_values()

turnover_records = []

# t0: Initial rebalance (full capital deployment from Cash)
t0_weights = df_bmk_target_constrained.loc[rebalance_dates[0]]
t0_turnover = 0.5 * t0_weights.abs().sum()

turnover_records.append(
    {
        "date": rebalance_dates[0],
        "turnover": t0_turnover,
        "transaction_cost_pct": t0_turnover
        * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL),
        "transaction_cost_bps": t0_turnover * TRANSACTION_COST_BPS_BASE,
    }
)

# t1 -> tN: Periodic rebalances against the drifted portfolio
for i in range(1, len(rebalance_dates)):
    prev_reb = rebalance_dates[i - 1]
    curr_reb = rebalance_dates[i]

    # Portfolio position right before rebalance (after market price drift)
    prev_weights = df_bmk_target_constrained.loc[prev_reb]
    window_rets = simple_returns.loc[
        (simple_returns.index > prev_reb) & (simple_returns.index <= curr_reb)
    ]
    compounded = (1.0 + window_rets).prod() - 1.0
    compounded = compounded.reindex(prev_weights.index).fillna(0.0)

    drifted = prev_weights * (1.0 + compounded)
    drifted_sum = drifted.sum()
    drifted_norm = (
        drifted / drifted_sum if drifted_sum > 0 else prev_weights
    )

    # New constrained target position at T
    curr_weights = df_bmk_target_constrained.loc[curr_reb]

    # Realized turnover
    realized_turnover = 0.5 * (curr_weights - drifted_norm).abs().sum()

    turnover_records.append(
        {
            "date": curr_reb,
            "turnover": realized_turnover,
            "transaction_cost_pct": realized_turnover
            * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL),
            "transaction_cost_bps": realized_turnover
            * TRANSACTION_COST_BPS_BASE,
        }
    )

df_turnover_schedule = pd.DataFrame(turnover_records).set_index("date")

# =============================================================================
# Aggregate Annualized Cost Impact (Base Scenario: 15 bps)
# =============================================================================

# Exclude t0 for steady-state annualized metrics calculation
mean_rebal_turnover = df_turnover_schedule["turnover"].iloc[1:].mean()
ann_turnover = mean_rebal_turnover * REBALANCINGS_PER_YEAR

ann_cost_pct = ann_turnover * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL)
ann_cost_bps = ann_turnover * TRANSACTION_COST_BPS_BASE

# =============================================================================
# Audit & Output Summary
# =============================================================================

print("=" * 80)
print("BENCHMARK B — TRANSACTION COST IMPACT AUDIT (BASE: 15 BPS)")
print("=" * 80)
print(f"✓ Rebalancing Dates Matched:        {len(rebalance_dates):,}")
print(f"✓ Start Rebalance Date:             {min(rebalance_dates):%Y-%m-%d}")
print(f"✓ End Rebalance Date:               {max(rebalance_dates):%Y-%m-%d}")
print("-" * 80)
print(f"✓ Average Turnover per Rebalance:   {mean_rebal_turnover:.2%}")
print(f"✓ Annualized Turnover:              {ann_turnover:.2%}")
print("-" * 80)
print(f"✓ Annual Transaction Cost (%):      {ann_cost_pct:.4%}")
print(f"✓ Annual Transaction Cost (bps):    {ann_cost_bps:.2f} bps")
print("=" * 80)

BENCHMARK B — TRANSACTION COST IMPACT AUDIT (BASE: 15 BPS)
✓ Rebalancing Dates Matched:        18
✓ Start Rebalance Date:             2025-01-15
✓ End Rebalance Date:               2026-06-18
--------------------------------------------------------------------------------
✓ Average Turnover per Rebalance:   32.58%
✓ Annualized Turnover:              390.92%
--------------------------------------------------------------------------------
✓ Annual Transaction Cost (%):      0.5864%
✓ Annual Transaction Cost (bps):    58.64 bps


La evaluación del **Benchmark B** bajo un marco de costes de transacción base de 15 puntos básicos (0,15% por operación) confirma un impacto operativo alineado con la naturaleza del factor *momentum* de corto plazo. Sobre el calendario homogéneo de 18 ventanas de rebalanceo (enero de 2025 a junio de 2026), la rotación media real por evento se sitúa en el **32,58%**, considerando la deriva de precios del mercado (*asset price drift*) entre periodos de negociación.

A escala anual, esta dinámica implica una rotación de cartera del **390,92%**, lo que genera una penalización por rozamiento operativo del **0,5864% anual (58,64 bps)** sobre la rentabilidad neta. La incorporación explícita del *drift* en el cálculo del *turnover* resulta fundamental: al medir la rotación frente al valor actualizado de las posiciones en lugar de sobre la asignación estática previa, se captura con precisión el ajuste real que debe ejecutar la mesa de negociación, permitiendo una comparación transparente y homogénea frente a la fricción operativa observada en los modelos de Machine Learning.

In [ ]:
from src.portfolio.metrics import calculate_benchmark_b_metrics
from src.portfolio.utils import format_comparison_table

# =============================================================================
# Load Baseline ML Configuration & Retrieve Precomputed Metrics
# =============================================================================

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)
df_metrics_all = pd.read_parquet(METRICS_PATH)

df_target_weights = pd.read_parquet(
    "../data/portfolio_results/candidate_target_weights.parquet"
)
df_temp = df_target_weights.reset_index()

# Filter target weights to extract the specific Baseline ML parameters
baseline_weights_mask = (
    (df_temp["role"] == "Baseline ML")
    & (df_temp["model"] == "Random Forest")
    & (df_temp["portfolio"] == "long_only_equal_weight")
    & (df_temp["frequency_days"] == 21)
)

df_baseline_config = df_temp[baseline_weights_mask]

if df_baseline_config.empty:
    raise ValueError(
        "Specified Baseline ML configuration not found in candidate_target_weights.parquet"
    )

# Extract key identifying values from the filtered weights
target_role = df_baseline_config["role"].iloc[0]
target_model = df_baseline_config["model"].iloc[0]
target_portfolio = df_baseline_config["portfolio"].iloc[0]
target_freq = df_baseline_config["frequency_days"].iloc[0]

# Query the precomputed metrics parquet using the retrieved baseline parameters
metrics_mask = (
    (df_metrics_all["model"] == target_model)
    & (df_metrics_all["portfolio"] == target_portfolio)
    & (df_metrics_all["frequency_days"] == target_freq)
)

df_baseline = df_metrics_all[metrics_mask].copy()

# =============================================================================
# 1. Align Daily Executed Weights & Returns to the Exact Same Date Index
# =============================================================================
# Intersection of trading dates between executed weights and returns matrix
common_dates = df_bmk_executed_daily.index.intersection(simple_returns.index)

# Reindex both dataframes to guarantee identical rows and columns
weights_aligned = df_bmk_executed_daily.loc[common_dates]
returns_aligned = simple_returns.loc[common_dates]

# =============================================================================
# 2. Compute Daily Gross Portfolio Returns
# =============================================================================
# Multiply yesterday's aligned weights by today's aligned simple returns
daily_gross_returns = (weights_aligned.shift(1) * returns_aligned).sum(axis=1)

# Drop the first row (t_0 shift creates a single NaN at the start)
daily_gross_returns = daily_gross_returns.dropna()

# =============================================================================
# 3. Integrate Transaction Costs (Aligned Dates)
# =============================================================================
# 3.1 Initialize zero series mapped to daily_gross_returns index
daily_costs_pct = pd.Series(0.0, index=daily_gross_returns.index)

# 3.2 Match rebalance dates directly against daily_gross_returns index
cost_dates_intersection = daily_gross_returns.index.intersection(
    df_turnover_schedule.index
)

# 3.3 Impute realized transaction costs on rebalancing days
daily_costs_pct.loc[cost_dates_intersection] = df_turnover_schedule.loc[
    cost_dates_intersection, "transaction_cost_pct"
]

# 3.4 Calculate Net Daily Returns
df_bmk_b_daily_returns = daily_gross_returns - daily_costs_pct

# Compute Benchmark B metrics
bmk_b_metrics_dict = calculate_benchmark_b_metrics(
    returns=df_bmk_b_daily_returns,
    ann_turnover_val=ann_turnover,
)

df_bmk_b = pd.DataFrame([bmk_b_metrics_dict])

# =============================================================================
# Combine & Format Final Comparison Table
# =============================================================================

target_cols = [
    "model",
    "portfolio",
    "frequency_days",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

# Align columns and concatenate
df_comparison = pd.concat(
    [df_bmk_b[target_cols], df_baseline[target_cols]], ignore_index=True
)

df_comparison_formatted = format_comparison_table(df_comparison)

# =============================================================================
# Print Standardized Audit Display
# =============================================================================

print("=" * 115)
print("BENCHMARK B vs BASELINE ML COMPARISON")
print("=" * 115)
print(df_comparison_formatted.to_string(index=False))
print("=" * 115)

BENCHMARK B vs BASELINE ML COMPARISON
          model              portfolio  frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
Momentum Top 10 long_only_max_cap_5pct              21 55.56%                31.51%             390.92%  1.763   2.539  2.344          -23.70%
  Random Forest long_only_equal_weight              21 72.42%                32.85%             233.38%  2.205   3.279  2.378          -30.45%


Al controlar las fricciones operativas y mantener la frecuencia de rebalancing en 21 días para ambas carteras, logramos aislar el impacto directo del modelo. La diferencia entre el **Benchmark B** (Momentum Top 10) y el **Baseline ML** (Random Forest) refleja estrictamente la capacidad predictiva del algoritmo frente a la simple tendencia del mercado.

El **Random Forest** demuestra una superioridad clara en la generación de Alpha: alcanza un CAGR del **72.42%** frente al **55.56%** del Benchmark B, añadiendo un rendimiento extra del **16.86%**. Esta ventaja no responde a una toma de riesgo desmedida, ya que la volatilidad se mantiene contenida en niveles equivalentes (**32.85%** frente a **31.51%**). Esto se traduce en una mejora notable del rendimiento ajustado por riesgo, elevando el ratio de **Sharpe** de **1.763 a 2.205** y el ratio de **Sortino** de **2.539 a 3.279**.

El modelo resulta ser además más eficiente operativamente: reduce el *annualized turnover* de un **390.92%** a un **233.38%**, lo que implica menor rotación, menor arrastre por costes de transacción y una selección de activos más estable en el tiempo.

La única contrapartida aparece en la gestión del riesgo de cola: el Random Forest registra un *Maximum Drawdown* mayor (**-30.45%** frente a **-23.70%**), mostrando cierta vulnerabilidad ante correcciones pronunciadas. Sin embargo, el **Calmar ratio** (**2.378** vs **2.344**) confirma que el exceso de retorno logrado compensa con creces esa mayor profundidad en la caída acumulada.

##### Benchmark C — Factor Momentum + Esquema de Ponderación Criterio (Signal Weighting)

Para aislar el impacto real de la señal del modelo frente a una regla cuantitativa tradicional, el Benchmark C replica de forma simétrica el esquema de ponderación seleccionado para las carteras finales. Si la estrategia candidata optima utiliza una asignación basada en la intensidad de la señal (Signal Weighting), el Benchmark C aplica exactamente ese mismo algoritmo sobre el factor de momentum mecánico a 21 días:

$$\text{Benchmark C} = \text{Momentum (Top 10\%)} \longrightarrow \text{Signal Weighting}$$

La justificación metodológica de este diseño radica en el principio de aislamiento estricto de variables. Introducir esquemas de optimización alternativos en este benchmark (como Risk Parity o Maximum Sharpe) alteraría simultáneamente la señal de entrada y el algoritmo de asignación de pesos, imposibilitando determinar el origen exacto del rendimiento ajustado por riesgo.

Al fijar la técnica de ponderación (Signal Weighting) de manera homogénea en ambas carteras, se elimina cualquier sesgo derivado de la construcción de la cartera. De este modo, la comparativa directa entre la estrategia candidata ($\text{ML} \rightarrow \text{Signal Weighting}$) y el Benchmark C ($\text{Momentum} \rightarrow \text{Signal Weighting}$) permite evaluar de forma limpia si la magnitud de la predicción generada por el modelo de Machine Learning aporta una información más valiosa y eficiente para escalar las posiciones que la simple intensidad del factor momentum tradicional.


In [28]:
# =============================================================================
# Step 1: Compute Target Weights (21-Day Momentum + Signal Weighting Top 10%)
# =============================================================================

# 1. Compute 21-day rolling cumulative returns (Momentum signal)
momentum_signal = adj_close.pct_change(MOMENTUM_LOOKBACK)

SIGNAL_WEIGHTING_K = 1.0
target_weights_list_bmk_c = []

for reb_date in rebalancing_dates:
    if reb_date not in momentum_signal.index:
        continue

    signals_at_date = momentum_signal.loc[reb_date].dropna()

    # Determine top 10% threshold count
    n_assets = len(signals_at_date)
    top_n = max(1, int(np.floor(n_assets * SELECTION_PERCENT)))

    # Select top N tickers with highest 21-day cumulative return
    top_tickers = signals_at_date.nlargest(top_n).index

    # Initialize raw weights aligned to simple_returns columns
    raw_weights = pd.Series(0.0, index=simple_returns.columns, name=reb_date)

    if top_n > 0:
        # Retrieve momentum signal values for selected universe
        selected_signals = signals_at_date.loc[top_tickers]

        # Re-rank within selected universe (0 to 1)
        selected_ranks = selected_signals.rank(ascending=True, method="first", pct=True)

        # Apply signal intensity exponent
        signal_strength = selected_ranks ** SIGNAL_WEIGHTING_K

        # Normalize weights to sum exactly 1.0 (100%)
        normalized_weights = signal_strength / signal_strength.sum()

        # Assign signal-weighted positions
        raw_weights[top_tickers] = normalized_weights

    target_weights_list_bmk_c.append(raw_weights)

# Consolidate raw target weights matrix for Benchmark C (Rebalance dates x Tickers)
df_bmk_c_target_raw = pd.DataFrame(target_weights_list_bmk_c)


# =============================================================================
# Step 2: Apply Constraints & Audit Benchmark C
# =============================================================================

from src.portfolio.benchmarks import apply_benchmark_constraints_with_drift

df_bmk_c_target_constrained, df_bmk_c_executed_daily = (
    apply_benchmark_constraints_with_drift(
        target_weights_wide=df_bmk_c_target_raw,
        simple_returns=simple_returns,
        max_turnover_design=MAX_TURNOVER_DESIGN,
        min_weight=MIN_WEIGHT,
        max_weight=MAX_WEIGHT,
    )
)

# Compute Realized Turnover considering Asset Drift
rebalance_dates_c = df_bmk_c_target_constrained.index
turnover_series_c = []

for i in range(1, len(rebalance_dates_c)):
    prev_reb = rebalance_dates_c[i - 1]
    curr_reb = rebalance_dates_c[i]

    # Portfolio before rebalance (drifted by market returns)
    prev_weights = df_bmk_c_target_constrained.loc[prev_reb]
    window_rets = simple_returns.loc[
        (simple_returns.index > prev_reb) & (simple_returns.index <= curr_reb)
    ]
    compounded = (1.0 + window_rets).prod() - 1.0
    compounded = compounded.reindex(prev_weights.index).fillna(0.0)

    drifted = prev_weights * (1.0 + compounded)
    drifted_sum = drifted.sum()
    drifted_norm = (
        drifted / drifted_sum if drifted_sum > 0 else prev_weights
    )

    # New target portfolio after rebalance constraints
    curr_weights = df_bmk_c_target_constrained.loc[curr_reb]

    # Realized turnover calculation
    realized_turnover = 0.5 * (curr_weights - drifted_norm).abs().sum()
    turnover_series_c.append(realized_turnover)

turnover_series_c = pd.Series(turnover_series_c, index=rebalance_dates_c[1:])

# Extract position weight statistics
active_target_weights_c = df_bmk_c_target_constrained.replace(0.0, np.nan)
active_executed_weights_c = df_bmk_c_executed_daily.replace(0.0, np.nan)


# =============================================================================
# Print Audit Report — Benchmark C
# =============================================================================
print("=" * 80)
print("BENCHMARK C — CONSTRAINTS & DRIFT AUDIT REPORT (SIGNAL WEIGHTING)")
print("=" * 80)

print(f"\n1. TURNOVER METRICS (accounting for Asset Price Drift):")
print(f"    • Rebalances evaluated: {len(turnover_series_c)}")
print(f"    • Max Turnover per event: {turnover_series_c.max():.2%}")
print(f"    • Mean Turnover per event: {turnover_series_c.mean():.2%}")
print(
    f"    • Annualized Turnover:      {turnover_series_c.mean() * REBALANCINGS_PER_YEAR:.2%}"
)

print(f"\n2. TARGET WEIGHTS BOX CONSTRAINTS (at Rebalance dates T):")
print(f"    • Max Position Weight:      {active_target_weights_c.max().max():.2%}")
print(f"    • Min Active Weight:        {active_target_weights_c.min().min():.2%}")

print(f"\n3. EXECUTED WEIGHTS BOX CONSTRAINTS (Daily T+1 with Drift):")
print(f"    • Max Position Weight:      {active_executed_weights_c.max().max():.2%}")
print(f"    • Min Active Weight:        {active_executed_weights_c.min().min():.2%}")
print("=" * 80)

BENCHMARK C — CONSTRAINTS & DRIFT AUDIT REPORT (SIGNAL WEIGHTING)

1. TURNOVER METRICS (accounting for Asset Price Drift):
    • Rebalances evaluated: 17
    • Max Turnover per event: 30.21%
    • Mean Turnover per event: 26.66%
    • Annualized Turnover:      319.94%

2. TARGET WEIGHTS BOX CONSTRAINTS (at Rebalance dates T):
    • Max Position Weight:      5.00%
    • Min Active Weight:        0.53%

3. EXECUTED WEIGHTS BOX CONSTRAINTS (Daily T+1 with Drift):
    • Max Position Weight:      7.72%
    • Min Active Weight:        0.33%


El diagnóstico del **Benchmark C** valida la correcta implementación técnica de la estrategia. La aplicación del esquema de *Signal Weighting* junto con la restricción de peso máximo por activo (*box constraint* del 5%) funciona exactamente según lo diseñado en las fechas de rebalanceo $T$.

El *annualized turnover* se sitúa en el **319.94%** (un **26.66%** medio por evento), lo que refleja un descenso importante en la rotación frente al Benchmark B (390.92%) gracias al escalado progresivo por intensidad de señal en lugar de la asignación uniforme. Por su parte, la derivación diaria (*drift*) en la ejecución real $T+1$ desplaza ligeramente los pesos activos hacia un abanico de entre **0.33% y 7.72%**, capturando de forma precisa el impacto del movimiento de los precios del mercado entre rebalanceos.

In [31]:
# =============================================================================
# Step 3: Compute Rebalance-Level Turnover (Accounting for Asset Drift)
# =============================================================================

TRANSACTION_COST_BPS_BASE = 15.0
BPS_TO_DECIMAL = 10_000.0

rebalance_dates_c = df_bmk_c_target_constrained.index.sort_values()
turnover_records_c = []

# t0: Initial rebalance (full capital deployment from Cash)
t0_weights_c = df_bmk_c_target_constrained.loc[rebalance_dates_c[0]]
t0_turnover_c = 0.5 * t0_weights_c.abs().sum()

turnover_records_c.append(
    {
        "date": rebalance_dates_c[0],
        "turnover": t0_turnover_c,
        "transaction_cost_pct": t0_turnover_c
        * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL),
        "transaction_cost_bps": t0_turnover_c * TRANSACTION_COST_BPS_BASE,
    }
)

# t1 -> tN: Periodic rebalances against the drifted portfolio
for i in range(1, len(rebalance_dates_c)):
    prev_reb = rebalance_dates_c[i - 1]
    curr_reb = rebalance_dates_c[i]

    # Portfolio position right before rebalance (after market price drift)
    prev_weights = df_bmk_c_target_constrained.loc[prev_reb]
    window_rets = simple_returns.loc[
        (simple_returns.index > prev_reb) & (simple_returns.index <= curr_reb)
    ]
    compounded = (1.0 + window_rets).prod() - 1.0
    compounded = compounded.reindex(prev_weights.index).fillna(0.0)

    drifted = prev_weights * (1.0 + compounded)
    drifted_sum = drifted.sum()
    drifted_norm = (
        drifted / drifted_sum if drifted_sum > 0 else prev_weights
    )

    # New constrained target position at T
    curr_weights = df_bmk_c_target_constrained.loc[curr_reb]

    # Realized turnover
    realized_turnover = 0.5 * (curr_weights - drifted_norm).abs().sum()

    turnover_records_c.append(
        {
            "date": curr_reb,
            "turnover": realized_turnover,
            "transaction_cost_pct": realized_turnover
            * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL),
            "transaction_cost_bps": realized_turnover
            * TRANSACTION_COST_BPS_BASE,
        }
    )

df_turnover_schedule_c = pd.DataFrame(turnover_records_c).set_index("date")

# =============================================================================
# Step 4: Compute Daily Gross and Net Returns (Aligned Stream)
# =============================================================================

# 4.1 Compute Gross Daily Returns with 1-day execution lag
weights_aligned_c = df_bmk_c_executed_daily.shift(1)
daily_gross_returns_c = (weights_aligned_c * simple_returns).sum(
    axis=1
).dropna()

# 4.2 Initialize zero series mapped directly to daily_gross_returns index
daily_costs_pct_c = pd.Series(0.0, index=daily_gross_returns_c.index)

# 4.3 Match rebalance dates directly against available return dates
cost_dates_intersection_c = daily_gross_returns_c.index.intersection(
    df_turnover_schedule_c.index
)

# 4.4 Impute realized transaction costs
daily_costs_pct_c.loc[cost_dates_intersection_c] = (
    df_turnover_schedule_c.loc[cost_dates_intersection_c, "transaction_cost_pct"]
)

# 4.5 Compute Net Daily Returns
df_bmk_c_daily_returns = daily_gross_returns_c - daily_costs_pct_c

# =============================================================================
# Step 5: Cost Impact Summary Report — Benchmark C
# =============================================================================

mean_rebal_turnover_c = df_turnover_schedule_c["turnover"].iloc[1:].mean()
ann_turnover_c = mean_rebal_turnover_c * REBALANCINGS_PER_YEAR
ann_cost_pct_c = ann_turnover_c * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL)
ann_cost_bps_c = ann_turnover_c * TRANSACTION_COST_BPS_BASE

print("=" * 80)
print("BENCHMARK C — TRANSACTION COST & ALIGNMENT REPORT")
print("=" * 80)
print(f"✓ Total Rebalance Events:           {len(df_turnover_schedule_c):,}")
print(f"✓ Imputed Cost Events (Intersect):  {len(cost_dates_intersection_c):,}")
print(f"✓ Backtest Start Date:              {min(rebalance_dates_c):%Y-%m-%d}")
print(f"✓ Backtest End Date:                {max(rebalance_dates_c):%Y-%m-%d}")
print("-" * 80)
print(f"✓ Cost Basis Assumption:            {TRANSACTION_COST_BPS_BASE:.1f} bps")
print(f"✓ Annual Transaction Cost (%):      {ann_cost_pct_c:.4%}")
print(f"✓ Annual Transaction Cost (bps):    {ann_cost_bps_c:.2f} bps")
print("=" * 80)

BENCHMARK C — TRANSACTION COST & ALIGNMENT REPORT
✓ Total Rebalance Events:           18
✓ Imputed Cost Events (Intersect):  18
✓ Backtest Start Date:              2025-01-15
✓ Backtest End Date:                2026-06-18
--------------------------------------------------------------------------------
✓ Cost Basis Assumption:            15.0 bps
✓ Annual Transaction Cost (%):      0.4799%
✓ Annual Transaction Cost (bps):    47.99 bps


In [33]:
from src.portfolio.metrics import calculate_benchmark_b_metrics
from src.portfolio.utils import format_comparison_table

# =============================================================================
# Load Top ML Models Configuration & Retrieve Precomputed Metrics
# =============================================================================

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)
df_metrics_all = pd.read_parquet(METRICS_PATH)

# Define target configurations for top performing ML models
top_models_config = [
    {
        "role": "Highest Sharpe",
        "model": "Random Forest",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest CAGR",
        "model": "XGBoost",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
]

# Filter precomputed metrics for each selected ML candidate
top_models_dfs = []
for config in top_models_config:
    mask = (
        (df_metrics_all["model"] == config["model"])
        & (df_metrics_all["portfolio"] == config["portfolio"])
        & (df_metrics_all["frequency_days"] == config["frequency_days"])
    )
    df_found = df_metrics_all[mask].copy()

    if df_found.empty:
        raise ValueError(
            f"Configuration {config['role']} ({config['model']} - {config['portfolio']}) not found in {METRICS_PATH}"
        )

    top_models_dfs.append(df_found)

df_top_models = pd.concat(top_models_dfs, ignore_index=True)

# =============================================================================
# Compute Benchmark C Metrics
# =============================================================================

# Compute metrics using net daily returns and turnover from Benchmark C
bmk_c_metrics_dict = calculate_benchmark_b_metrics(
    returns=df_bmk_c_daily_returns,
    ann_turnover_val=ann_turnover_c,
)

# Override identifiers to explicitly label Benchmark C
bmk_c_metrics_dict["model"] = "Momentum (21d)"
bmk_c_metrics_dict["portfolio"] = "benchmark_c_top_10_signal_weight"
bmk_c_metrics_dict["frequency_days"] = 21

df_bmk_c = pd.DataFrame([bmk_c_metrics_dict])

# =============================================================================
# Combine & Format Final Comparison Table
# =============================================================================

target_cols = [
    "model",
    "portfolio",
    "frequency_days",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

# Concatenate Benchmark C alongside the top ML candidates
df_comparison_c = pd.concat(
    [df_bmk_c[target_cols], df_top_models[target_cols]], ignore_index=True
)

df_comparison_c_formatted = format_comparison_table(df_comparison_c)

# =============================================================================
# Print Standardized Audit Display
# =============================================================================

print("=" * 115)
print("BENCHMARK C vs TOP ML MODELS COMPARISON (SIGNAL WEIGHTING)")
print("=" * 115)
print(df_comparison_c_formatted.to_string(index=False))
print("=" * 115)

BENCHMARK C vs TOP ML MODELS COMPARISON (SIGNAL WEIGHTING)
         model                        portfolio  frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
Momentum (21d) benchmark_c_top_10_signal_weight              21  4.73%                10.42%             319.94%  0.454   0.655  0.182          -26.03%
 Random Forest   long_only_top_10_signal_weight              21 89.75%                36.28%             219.69%  2.474   3.663  2.764          -32.47%
       XGBoost   long_only_top_10_signal_weight              21 81.63%                35.35%             234.84%  2.309   3.412  2.522          -32.37%



#### Benchmark D — ML + Equal Weight (Evaluación de la Ponderación de Cartera)

Se fija la selección de activos derivada de la predicción ML y se compara la cartera equiponderada frente a los esquemas sofisticados de asignación de pesos (*Signal Weighting*, *Risk Parity*, *Maximum Sharpe*):

$$\text{Benchmark D} = \text{ML (Top 10\%)} \longrightarrow \text{Equal Weight} \quad \text{vs.} \quad \text{ML (Top 10\%)} \longrightarrow \text{Optimizer}$$

Esta prueba cuantifica exactamente la fracción de retorno ajustado por riesgo que aporta el módulo de construcción de carteras respecto a una asignación pasiva no informada.





### 7.3 Statistical Significance: Random Selection Benchmark (Monte Carlo)

Incluso si una estrategia supera al S&P 500 y a los *benchmarks* factoriales, persiste el riesgo de que la selección de activos por parte del modelo sea el resultado de un acierto aleatorio dentro del espacio de búsqueda o de la captura fortuita de ruido *out-of-sample*.

Para verificar la presencia de una capacidad predictiva genuina de forma estadísticamente rigurosa, se implementa una prueba de hipótesis empírica mediante la **Simulación de Monte Carlo**.



#### 7.3.1 Metodología del Test de Selección Aleatoria

Se simula el comportamiento de $N = 500$ carteras pseudo-aleatorias construidas mediante un proceso estocástico que preserva de forma estricta las restricciones operativas de la estrategia real:

* **Mismo universo y dimensión:** En cada fecha de rebalanceo $t_k$, se seleccionan al azar $K$ activos del universo elegible (donde $K$ coincide exactamente con el número de activos elegidos por el modelo ML, ej. $10\%$ del S&P 500).
* **Misma frecuencia y calendario:** El rebalanceo se ejecuta respetando la misma ventana temporal ($\Delta t = 21$ días) y con el desfase operativo de ejecución ($t_k + 1$).
* **Mismo esquema de ponderación y fricción:** A los activos seleccionados aleatoriamente se les aplica el mismo método de asignación de pesos y se descuenta el coste de transacción base ($15\text{ bps}$).

#### 7.3.2 Construcción de la Distribución Empírica y P-Valor

Para cada simulación $j \in \{1, \dots, N\}$, se calcula la tasa de crecimiento anual compuesta neta, obteniendo la distribución nula de rentabilidades:

$$\left\{ \text{CAGR}_{1}^{\text{random}}, \text{CAGR}_{2}^{\text{random}}, \dots, \text{CAGR}_{N}^{\text{random}} \right\}$$

El **p-valor empírico** de la estrategia ML se define como la proporción de carteras aleatorias que lograron igualar o superar el rendimiento neto de la estrategia candidata ($\text{CAGR}_{\text{ML}}$):

$$p_{\text{empírico}} = \frac{1}{N} \sum_{j=1}^{N} \mathbb{I}\left( \text{CAGR}_{j}^{\text{random}} \ge \text{CAGR}_{\text{ML}} \right)$$

donde $\mathbb{I}(\cdot)$ representa la función indicadora.

Si la estrategia de *Machine Learning* se sitúa en un percentil empírico superior al $95\%$ ($p_{\text{empírico}} < 0.05$), se rechaza la hipótesis nula de selección fortuita, confirmando empíricamente que el modelo extrae un patrón informacional real que supera al azar de forma estadísticamente significativa.

### 7.4 Risk-Adjusted Active Metrics

Una vez demostrada la significación estadística, la evaluación de la gestión activa requiere transformar la comparación de rentabilidades absolutas en una cuantificación del **riesgo activo** asumido para generar dicho exceso de retorno.

Para cada estrategia candidata se calcula la serie temporal de retornos diferenciales diarios respecto al *benchmark* de referencia ($R_{b, t}$):

$$R_{\text{active}, t} = R_{\text{strategy}, t} - R_{b, t}$$

A partir de esta serie de retorno activo, se derivan tres métricas fundamentales de la teoría cuantitativa de carteras:



#### 7.4.1 Excess CAGR

Mide la diferencia directa entre la tasa de crecimiento anual compuesta de la estrategia y la del *benchmark*:

$$\text{Excess CAGR} = \text{CAGR}_{\text{strategy}} - \text{CAGR}_{\text{benchmark}}$$

#### 7.4.2 Tracking Error (TE)

Representa la desviación estándar anualizada de la serie de retornos diferenciales diarios. Cuantifica la volatilidad de las decisiones de desviación que toma la estrategia respecto al índice de referencia:

$$\text{Tracking Error} = \sigma\left( R_{\text{active}} \right) \times \sqrt{252} = \sqrt{\frac{252}{T-1} \sum_{t=1}^{T} \left( R_{\text{active}, t} - \bar{R}_{\text{active}} \right)^2}$$

Un *Tracking Error* elevado indica una cartera marcadamente desalineada de la estructura del *benchmark*, lo que implica la asunción de un riesgo estructural independiente del mercado.

### 7.4.3 Information Ratio (IR)

Es el indicador central de eficiencia en la gestión activa. Mide el exceso de rentabilidad anualizado generado por cada unidad de riesgo activo (*Tracking Error*) asumido:

$$\text{Information Ratio} = \frac{\bar{R}_{\text{strategy, ann}} - \bar{R}_{b, \text{ann}}}{\text{Tracking Error}} = \frac{\left( \bar{R}_{\text{strategy}} - \bar{R}_{b} \right) \times 252}{\sigma\left( R_{\text{active}} \right) \times \sqrt{252}}$$

A diferencia del *Sharpe Ratio* (que penaliza el riesgo total), el *Information Ratio* aísla la habilidad del gestor para remunerar las desviaciones del índice base. Valores de $\text{IR} > 0.50$ se consideran representativos de una estrategia activa sólida, mientras que valores de $\text{IR} > 1.00$ reflejan un nivel excepcional de generación de alfa ajustado por riesgo activo.



### 7.5 Temporal Consistency & Rolling Performance

El cálculo de métricas agregadas sobre un periodo *out-of-sample* completo corre el riesgo de ocultar la inestabilidad temporal de un modelo. Una estrategia puede presentar métricas globales atractivas gracias a un comportamiento extraordinariamente positivo en un único año puntual que compense periodos prolongados de bajo rendimiento o estancamiento.

Para garantizar que la rentabilidad deviene de un alfa consistente y no de eventos aislados, esta sección aplica un análisis continuo de dinamismo temporal.



#### 7.5.1 Rolling 36-Month Sharpe Ratio

Se calcula el *Sharpe Ratio* en ventanas móviles superpuestas de $36$ meses ($756$ sesiones de negociación). Para cada día $t \ge 756$, la métrica se define como:

$$\text{Sharpe}_{36\text{m}, t} = \frac{\bar{R}_{t-756:t} - R_f}{\sigma\left( R_{t-756:t} \right)} \times \sqrt{252}$$

La representación gráfica de esta serie temporal permite evaluar la suavidad en la generación de retorno ajustado por riesgo, detectando fases de degradación del modelo, cambios de régimen de mercado o pérdidas de capacidad predictiva a medida que la muestra evoluciona.



#### 7.5.2 Continuous Drawdown Profile

En lugar de resumir el riesgo de caída en una única cifra estática (*Maximum Drawdown*), se mapea de forma continua la trayectoria de pérdida acumulada desde el máximo histórico anterior ($W_t$ representa el valor patrimonial acumulado en la fecha $t$):

$$DD_t = \frac{W_t}{\max_{s \le t} W_s} - 1, \quad \forall t \in [1, T]$$

Esta serie temporal visibiliza con total precisión la profundidad de las caídas patrimoniales, la frecuencia de las etapas de pérdida y la duración exacta de los periodos de recuperación (*underwater duration*), permitiendo comparar la resiliencia de las carteras candidatas frente al S&P 500 durante episodios reales de tensión en los mercados.



#### 7.5.3 Annual Returns Breakdown (Calendar Year Performance)

Finalmente, se desagrega el rendimiento neto de las carteras candidatas y de los *benchmarks* por años naturales completados en la muestra *out-of-sample*. Esta descomposición en formato tabla y mapa de calor (*heatmap*) verifica la consistencia interanual de la estrategia, confirmando si la tasa de éxito del sistema se mantiene homogénea a lo largo del horizonte de inversión.